# Musanga V4 — NB05 V4.018 | Harmonized modality experiment (Option B)

**Purpose:** isolate the added value of structural (DSM) and multispectral (MS) information for Musanga crown segmentation and fractional-cover estimation. This is a *new, independent experiment*; NB02/NB03/NB04 checkpoints and outputs remain immutable.

## Pre-registered primary comparison

| Arm | Features | Backbone and initialization | Validation |
|---|---|---|---|
| M0 | RGB (3 channels) | ResNet18 U-Net, frozen NB02 V4.007 baseline | Original NB01 spatial folds |
| M1 | RGB + DSM (same predefined structural feature stack) | **Same ResNet18 U-Net** initialized from NB02; added input weights zero | Same outer/inner folds, loss, optimization, stopping |
| M2 | RGB + MS (red, green, red-edge, NIR) | **Same ResNet18 U-Net** initialized from NB02; added input weights zero | Same outer/inner folds, loss, optimization, stopping |

**Interpretation:** M0 is the saved RGB backbone, held fixed for direct pairing. For an entirely symmetric *from-scratch training* comparison, M0 must be retrained with exactly the same recipe (a later, separately labeled arm); do not claim identical original optimizer settings without verifying NB02.

**Primary estimand:** paired change in *plot-level* Musanga fractional-cover absolute error (percentage points), calculated from full original reference-unit spatial hold-outs on the **identical RGB–DSM–MS valid footprint**. Report R², RMSE, bias, F1/IoU and MOS high-cover (>40%) separately. Also report RGB–DSM and RGB–MS on their larger respective matched footprints, not as direct three-way comparisons.

**Fold definition:** the immutable NB01 spatial CV assignment of the same 78 field plots and 34 complete circles, with exclusion rules and spatial guards from NB01. Each eligible reference unit is predicted once out of fold. Three folds are a computationally tractable *partition of spatial held-out units*, not three scientific replicates, and not leave-one-entire-site-out.

**Yoko:** include the 2020 Carlier P1/P2 MS coverage in *the same folds as every other site*. A valid training fold may contain Yoko labelled chips **only if Yoko is in the outer-training partition**; held-out Yoko labels must never leak into training. P1 and P2 must not be spliced band-by-band into a single observation. Missing MS coverage is missing data, not negative Musanga.

## Stage 1 — QA and frozen experimental contract (this notebook)

Run **00 → 01 → 02 → 03 → 04 → 05**. These cells inspect raster headers and reference geometry only, and produce the *candidate* modality/plot support contract. **They do not train models**. Review the Yoko filename/band matches, footprint overlap, and crown-scale coregistration before permitting expensive Stage 2 fitting.

The subsequent trainer must use the same spatial manifests, sampling seeds, augmentation, batch sampler, 256-pixel windows, training loss, inner-site-macro BCE selection and fixed 0.5 prediction threshold for M1 and M2, with frozen RGB as an eligible fallback.

### V4.006 interpretation

Yoko P1/P2 band discovery succeeded (8 expected spectral bands). The original three-fold assignments remain frozen. MOS and Yangambi have some incomplete MS footprint; Yoko fold 2 has no candidate plot MS coverage. The candidate coverage table counts a reference unit separately per intersecting RGB mosaic, and its `n_common_candidate` can exceed the distinct `n_units`; this is not additional independent validation. V4.006 adds a **bounded sampled pixel-validity audit** and explicit unit deduplication. This is not yet an exhaustive mask or a crown-scale co-registration certificate; no new models are trained.

### V4.006 — Exhaustive sensor support before harmonized training

**Why:** The previous 13 × 32 × 32 sampled interior windows yielded universally 100% overlap; this is not evidence that complete reference plots have complete coverage. Sections **06 and 07** now count actual valid pixels over every complete original plot and eligible circle, **including no-MS units**, using bounded raster windows. Each flight is kept separate, and multispectral coverage is the union of *complete* four-band acquisitions. No cross-flight mixing of channels is allowed.

**Fast run:** 00 → 01 → **06 → 07**. The earlier 02–05 candidate audits may be skipped if you only need the exact coverage. Existing images and checkpoints are read-only. This checks spectral *support*, not geometric co-registration accuracy or DSM multiscale filter halos; these still require separate QA before training.

**V4.006 reporting correction:** Yoko is one site and one four-band MS modality. P1 and P2 are only two coverage acquisitions; their combined valid-pixel **union** is reported for each original Yoko plot/circle. No per-acquisition performance or coverage tables. The V4.003 exact-coverage cache is reused unchanged to avoid re-reading rasters.

**V4.006** adds explicit DSM TIFF header and bounded pixel read preflight to section 06. A corrupt/incomplete DSM (e.g. `Mos_1_2_dsm_georef.tif: TIFFReadDirectory`) stops with file-level diagnostics; it is never misrepresented as missing geographical support. Prior completed unit records in the V4.003 coverage cache are retained. Do not re-export, copy or replace files automatically; verify the source first.

## V4.006 — Option B actual training implementation

**Run 00 → 01 → 08 → 09 → 10 with GPU (T4 or better)**. Sections 02–07 are earlier coverage diagnostics and need not rerun. The main RGB–DSM–MS common-support U-Net training deliberately uses original NB02 source-labelled **training** chips, the **frozen NB03 inner blocked split**, and the same reference units, masks, and training recipe across modalities. The old RGB checkpoint is held fixed as a model anchor, not a claim that NB02 and new fusion models had identical historical optimizer settings. Sections 08–10 do NOT use outer-held-out reference labels. Separate full-unit evaluation is the next stage.

**Important:** The combined MS Yoko acquisitions are one site: select a *single complete four-band acquisition per chip* by valid support, never mix P1/P2 channels. For the primary matched three-way comparison, all methods receive exactly the same valid pixels. DSM input has the original NB03 10-feature multiscale recipe, normalized using inner-training chips only. Fusion checkpoints are per fold and modality with provenance and no overwrite of earlier experiments.

**V4.008 scientific correction:** strict four-arm RGB / RGB+DSM / RGB+MS / RGB+DSM+MS fine-tuning on IDENTICAL common-valid source-labelled chips, optimizer, inner fold split, model initialization, and best **trained** (epoch ≥1) validation checkpoint. The original NB02 RGB remains initialization for ALL four arms, NOT the comparator. Earlier V4.006/007 checkpoints are intact and are not used for primary comparison. No fallbacks or selections reverting to RGB. Run 00 → 01 → 08 → 10 → 11 → 12. 08A caches are reused; no full extraction required.

**V4.009 quality fix:** section 11 includes the needed spectral VRT and tile extraction helper functions, so running 08A again is unnecessary. Output checkpoints still use the scientifically harmonized V4.008 training contract. Starting this notebook from a fresh session: 00 → 01 → 08 → 10 → 11 → 12. Existing expensive joint-chip caches and successfully finished V4.008 trained checkpoints are reused.

,**V4.010 performance revision:** block 11 evaluates 512-pixel blocks once per mosaic/fold, distributing the four-model prediction to every intersecting original reference unit. It saves atomic block ledgers and per-unit accumulators after each five blocks, to resume without redoing completed blocks after an interruption. Original fractional-cover and segmentation metrics, fixed outer folds, matched masks and checkpoints are preserved. Runs with 00 → 01 → 08 → 10 → 11 → 12; completed four-arm checkpoints and chip caches remain reusable. The new inference output folder is independent of earlier runs.

**V4.012 speed patch:** 256px tile prediction LRU cache (160 tiles, approx 170 MB RAM) within each fold/mosaic, reused across adjacent 512px blocks. All four model predictions, DSM structural extraction, MS acquisition choice, pixel masks, Hann blend and spatial folds are *identical* to V4.011. The V4.011 block checkpoint folder and signature are unchanged; stopping V4.011 after a checkpoint and restarting V4.012 resumes completed blocks rather than discarding the long Yangambi run. Cache exists in RAM only and clears on next mosaic/fold. Reports tile reuse in progress logs. Expect a useful but workload-dependent acceleration, not a guaranteed 4× gain.

**V4.013 resource-aware I/O improvement:** optional, source-verified SSD staging of active mosaic RGB/DSM/MS files using the ample local Colab storage; larger GPU batch 12 (from 6); retains original 512px block grid, 256/128 overlap, DSM feature recipe, source-data versions, old completed block checkpoints, and scientific metrics. Data transfers may have an initial cost; only beneficial if Drive random-access overhead dominates. Scratch files live under /content and disappear on runtime reset. The stage can be disabled with ENABLE_SSD_STAGE11=False. Do not run V4.012 and V4.013 block 11 simultaneously on the same ledger.

**V4.014 critical speed fix:** inspect cached block keys and RGB header *before* staging. Completed mosaics return immediately without copying any RGB/DSM/MS files. Partially completed mosaics still reuse existing data and checkpoints. Older V4.013 stage downloads remain in local SSD in this Colab runtime; do not clear them. All numerical inference and output directories remain unchanged. When a newly encountered mosaic has large files and few blocks, optional staging may still be slower; use ENABLE_SSD_STAGE11=False to disable.

V4.015 adds section 13: a one-block, read-only Yangambi profiler. Paste/run only section 13 in the existing stopped Colab session; do NOT run section 11 again to prepare it. Existing block ledgers remain unchanged.

V4.016 extends the read-only section 13 profiler with two passes on the same 512px block, a cold-vs-warm timing comparison and numerical/mask reproducibility checks. No file or block checkpoint is written.

**V4.017 adds section 14**: two-tile DSM micro-profiler separating high-resolution local DSM filters from coarse multiscale context. Read-only, same current Colab session, no copying, training or block progress. Section 13 already established DSM occupies 87.6% of inference time. The micro-profile determines which part to optimize without changing scientific feature definitions.

V4.018 adds section 15: benchmark replacing seven independent contextual-band GDAL resampling calls with one 7-band warp. Checks numeric feature equivalence on four Yangambi tiles, without changing main production functions, saved evaluation or GPU model weights. Run ONLY section 15 in initialized stopped Colab session.


## 00 — Read-only environment and immutable reference contract


In [ ]:
from pathlib import Path
import os, re, json, hashlib
from collections import defaultdict
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from shapely.geometry import box
from rasterio.warp import transform_bounds
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DATA=ROOT/'Data'
REF=ROOT/'Outputs/v4/01_reference_field78_combined'
N0=ROOT/'Outputs/v4/00_input_audit'
OUT=ROOT/'Outputs/v4/05_harmonized_optionB_v4_003'
OUT.mkdir(parents=True,exist_ok=True)
CONTRACT=json.loads((REF/'01_field78_reference_contract.json').read_text())
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
FOLDS=pd.read_csv(CONTRACT['folds_csv'])
CF=pd.read_csv(REF/'01_field78_circle_folds.csv')
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs('EPSG:32635')
CIRCLES=gpd.read_file(NB00['circle_complete_roi']).to_crs('EPSG:32635')
assert len(PLOTS)==78 and len(CIRCLES)==34
assert FOLDS.source_id.is_unique and set(FOLDS.cv_fold)=={0,1,2}
assert set(CF.cv_fold)=={0,1,2} and len(CF)==34
PLOTS=PLOTS.merge(FOLDS[['source_id','site_key','cv_fold']],left_on='Plot_name',
                  right_on='source_id',validate='one_to_one')
CIRCLES['circle_id']=np.arange(len(CIRCLES))
CIRCLES=CIRCLES.merge(CF[['circle_id','site_key','cv_fold',
    'cross_fold_plot_intersection']],on='circle_id',validate='one_to_one')
ROI=gpd.read_file(DATA/'Ortho_RGB/ROI.shp').to_crs('EPSG:32635')
CV=pd.read_csv(CONTRACT['cv_manifest'])
assert not CV.duplicated(['cv_fold','mosaic','row','col','width','height']).any()
assert set(CV.cv_fold)=={0,1,2}
print('Frozen plot counts by site/fold')
display(PLOTS.groupby(['site_key','cv_fold']).size().unstack(fill_value=0))
print('Frozen circles by site/fold')
display(CIRCLES.groupby(['site_key','cv_fold']).size().unstack(fill_value=0))
print('Frozen window manifest',len(CV),'rows')


## 01 — Sensor registry including Yoko P1/P2 (headers only)


In [ ]:
# Historical exact RGB–DSM pairing is read from NB00; do not guess DSM filenames.
PAIRS=pd.read_csv(N0/'00_rgb_dsm_pair_audit.csv')
assert len(PAIRS)==6 and PAIRS.rgb.is_unique
RGB_DIR=DATA/'Ortho_RGB'
REG=pd.read_csv(N0/'00_ms_raster_headers.csv')
REG=REG.loc[REG.error.fillna('').eq('')]
MS_BANDS={}
RGB_TO_SITE={}
for record in PAIRS.itertuples(index=False):
    p=Path(record.rgb_path)
    assert p.is_file() and Path(record.dsm_path).is_file()
    RGB_TO_SITE[p.name]=str(next(iter(CV.loc[CV.mosaic.eq(p.name),'site'].unique())))
assert len(RGB_TO_SITE)==6
# Preserve exact historical four-band selection; P1/P2 are added separately.
BASE_GROUPS=['Baego_MS','MOS_1_2_MS','MOS_3_4_MS','MOS_5_6_MS',
             'Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS']
MOSAIC_KEY={
'Baego_MS':'Baego.tif',
'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
for group in BASE_GROUPS:
    part=REG[REG.group.eq(group)]
    if group=='Baego_MS':
        assert len(part)==1,(group,len(part))
        p=Path(part.iloc[0].path)
        with rasterio.open(p) as src:
            assert src.count>=4
        MS_BANDS[group]={b:p for b in ('red','green','red_edge','nir')}
    else:
        files={}
        for b in ('red','green','red_edge','nir'):
            matches=part.loc[part.band_hint.eq(b)]
            if b=='red_edge' and group=='MOS_1_2_MS' and len(matches)>1:
                matches=matches.loc[matches.name.eq(
                    'mlt_Mos_1_2_transparent_mosaic_red edge_georef.tif')]
            assert len(matches)==1,(group,b,list(matches.name))
            files[b]=Path(matches.iloc[0].path)
        MS_BANDS[group]=files
# Enumerate only the new Yoko names, with no image pixel reads.
yoko_root=DATA
yoko_folders=[]
for folder,dirs,files in os.walk(yoko_root):
    for d in dirs:
        if 'carlier2020_yoko_micasense' in d.lower():
            yoko_folders.append(Path(folder)/d)
    dirs[:]=[d for d in dirs if d not in
        {'Outputs','.git','__pycache__'}]
# Folders may also be dataset filename prefixes rather than directory names.
raw=[]
for p in yoko_folders:
    raw.extend(q for q in p.rglob('*') if q.is_file() and q.suffix.lower() in ('.tif','.tiff'))
# Support plain TIFF filenames stored directly in any DATA subdirectory.
for folder,dirs,files in os.walk(yoko_root):
    for name in files:
        if ('carlier2020_yoko_micasense' in name.lower() and
            name.lower().endswith(('.tif','.tiff'))):
            raw.append(Path(folder)/name)
    dirs[:]=[d for d in dirs if d not in {'Outputs','.git','__pycache__'}]
raw=sorted(set(raw),key=str)
def classify(name):
    s=re.sub('[^a-z0-9]+','_',name.lower())
    if re.search(r'(^|_)(red_edge|rededge|red_?edge)(_|$)',s):return 'red_edge'
    if re.search(r'(^|_)(nir|near_infrared)(_|$)',s):return 'nir'
    if re.search(r'(^|_)green(_|$)',s):return 'green'
    if re.search(r'(^|_)red(_|$)',s):return 'red'
    return None
YOKO={}
for flight in ('P1','P2'):
    matches=[]
    for p in raw:
        s=str(p).lower()
        if re.search(r'(^|[_/\\-])'+flight.lower()+r'($|[_/\\.-])',s):
            # filename first, then parent band-labeled folder
            band=classify(p.stem) or classify(p.parent.name)
            if band:matches.append((band,p))
    found=defaultdict(list)
    for band,p in matches:found[band].append(p)
    for band in ('red','green','red_edge','nir'):
        if len(found[band])!=1:
            print('YOKO',flight,band,'candidates',list(map(str,found[band])))
    if all(len(found[b])==1 for b in ('red','green','red_edge','nir')):
        group=f'Yoko_Carlier2020_MicaSense_{flight}_MS'
        YOKO[group]={b:found[b][0] for b in ('red','green','red_edge','nir')}
        yoko_rgb=[p for p in RGB_DIR.glob('*.tif') if 'yoko' in p.name.lower()]
        assert len(yoko_rgb)==1,('Yoko RGB mosaic ambiguity',yoko_rgb)
        MOSAIC_KEY[group]=yoko_rgb[0].name
        MS_BANDS[group]=YOKO[group]
    else:
        print('YOKO FLIGHT NOT REGISTERED',flight,'manual filename validation required')
print('DISCOVERED YOKO SPECTRAL FILES:',len(raw))
display(pd.DataFrame([dict(flight=k,band=b,path=str(p)) for k,ff in YOKO.items()
                     for b,p in ff.items()]))
assert len(YOKO)==2,'Need 4 unambiguous first-band TIFFs for each P1 and P2 before training'
HEADERS=[]
for group,files in MS_BANDS.items():
    for band,p in files.items():
        with rasterio.open(p) as src:
            assert src.count>=1 and src.crs is not None,(group,band,p)
            index=('red','green','red_edge','nir').index(band)+1 if group=='Baego_MS' else 1
            assert src.count>=index
            HEADERS.append(dict(group=group,band=band,filename=p.name,
                 data_band=index,path=str(p),crs=str(src.crs),width=src.width,
                 height=src.height,bounds=str(src.bounds),pixel_m=abs(src.transform.a)))
HEADERS=pd.DataFrame(HEADERS)
HEADERS.to_csv(OUT/'05_harmonized_sensor_registry.csv',index=False)
display(HEADERS[['group','band','filename','data_band','pixel_m']])


## 02 — Pairwise and common-support candidate manifest


In [ ]:
# BOUNDING BOX overlap is a candidate only. True MS-valid, DSM-valid pixel
# footprints must be evaluated in later bounded window QA before training.
DSM_BY_RGB={Path(z.rgb_path).name:Path(z.dsm_path) for z in PAIRS.itertuples(index=False)}
U=[]
for kind,frame,idcol in [('plot',PLOTS,'Plot_name'),('circle',CIRCLES,'circle_id')]:
    for q in frame.itertuples(index=False):
        if kind=='circle' and bool(q.cross_fold_plot_intersection):continue
        geo=q.geometry
        if geo is None or geo.is_empty:continue
        possible=[m for m in set(MOSAIC_KEY.values()) if m in DSM_BY_RGB
                  and m in RGB_TO_SITE and RGB_TO_SITE[m]==q.site_key]
        for mosaic in possible:
            with rasterio.open(RGB_DIR/mosaic) as ds, rasterio.open(DSM_BY_RGB[mosaic]) as sd:
                geometry=geo.intersection(box(*ds.bounds))
                if geometry.is_empty:continue
                rgeom=box(*transform_bounds(sd.crs,ds.crs,*sd.bounds))
                dsm_area=geometry.intersection(rgeom).area
                acquisitions=[g for g,m in MOSAIC_KEY.items() if m==mosaic]
                footprints=[]
                for group in acquisitions:
                    bboxes=[]
                    for path in set(MS_BANDS[group].values()):
                        with rasterio.open(path) as band:
                            bboxes.append(box(*transform_bounds(
                                band.crs,ds.crs,*band.bounds,densify_pts=21)))
                    # intersection of ALL FOUR band bounding boxes, not union
                    intersection=geometry
                    for b in bboxes:intersection=intersection.intersection(b)
                    footprints.append(intersection)
                from shapely.ops import unary_union
                ms_region=unary_union(footprints) if footprints else geometry.intersection(box(0,0,0,0))
                common=ms_region.intersection(rgeom)
                U.append(dict(kind=kind,unit_id=str(getattr(q,idcol)),site=q.site_key,
                     fold=int(q.cv_fold),mosaic=mosaic,plot_area_m2=geo.area,
                     rgb_bbox_area_m2=geometry.area,dsm_bbox_area_m2=dsm_area,
                     ms_bbox_area_m2=ms_region.area,
                     common_bbox_area_m2=common.area,
                     common_bbox_fraction=common.area/geo.area))
UNIT_CANDIDATES=pd.DataFrame(U)
assert not UNIT_CANDIDATES.empty
UNIT_CANDIDATES.to_csv(OUT/'05_candidate_unit_modality_support.csv',index=False)
print('Candidate unit overlap, NOT confirmed valid-pixel support:')
display(UNIT_CANDIDATES.groupby(['site','fold','kind']).agg(
    n_units=('unit_id','nunique'),
    n_common_candidate=('common_bbox_area_m2',lambda x:int((x>0).sum())),
    median_common_bbox_frac=('common_bbox_fraction','median')).round(3))


## 03 — Freeze the Option B protocol and stop before expensive fitting


In [ ]:
# Stage 1 output is a provenance-audited *candidate* contract, not a final
# scientific validation mask. No global image reads or new model training.
import datetime
protocol=dict(
  name='NB05 Option B harmonized modality comparison',
  status='STAGE1_CANDIDATE_SUPPORT_NOT_PIXEL_VALIDATED',
  frozen_cv_manifest=str(CONTRACT['cv_manifest']),
  frozen_plot_folds=str(CONTRACT['folds_csv']),
  frozen_circle_folds=str(REF/'01_field78_circle_folds.csv'),
  dataset_sites=['Baego','MOS','Yangambi','Yoko'],
  outer_folds=[0,1,2],tile_px=256,holdout_unit='original plot/circle',
  primary_pairing='identical RGB+DSM+four-band MS valid pixels per unit',
  mandatory_training=['same ResNet18 U-Net backbone',
      'same original NB02 initialized RGB encoder and decoder weights',
      'zero-initialize additional input weights',
      'identical inner split (blocked; guard; exhaustive-only model selection)',
      'identical batch/site sampling, augmentation, loss, optimizer, LR, training steps, patience',
      'fold-safe normalization fit on training chips only',
      'no site holdout mixing; no outer-label access during selection'],
  modality_arms={'M0':'RGB3 frozen historical baseline',
       'M1':'RGB3+predefined DSM features early fusion',
       'M2':'RGB3+relative four-band MS early fusion'},
  planned_primary_metrics=['paired plot cover MAE difference in pp',
       'cover R2','RMSE','bias','pixel IoU/F1','MOS >40% bias'],
  exclusion='No modality missingness imputation or cross-flight band mosaicking',
  stage1_sensor_registry=str(OUT/'05_harmonized_sensor_registry.csv'),
  stage1_candidate_support=str(OUT/'05_candidate_unit_modality_support.csv'))
target=OUT/'05_optionB_frozen_protocol_STAGE1.json'
target.write_text(json.dumps(protocol,indent=2))
print('SAVED',target)
print('Stage 1 is complete. Do NOT train until exact MS, DSM mask and crown-scale alignment QA is reviewed.')


## 04 — Bounded actual-pixel RGB / DSM / MS overlap audit


In [ ]:
# 04 — Windowed RGB/DSM/MS actual pixel validity screening, NO whole raster reads.
# Sample several tiny windows per original unit on each intersecting RGB mosaic.
# Output is an estimated fraction on sampled in-polygon pixels, NOT exhaustive.
from contextlib import ExitStack
from shapely.geometry import Point
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.features import rasterize
from collections import defaultdict
import rasterio, hashlib, json, os
PATCH04=32
MAX_POINTS04=13
MS4=('red','green','red_edge','nir')
SAMPLE_ALGO04='optionB_v002_13spatialpoints_32px_RGB_DSM_MS4_v1'
COVER_DIR04=OUT/'05_pixel_sampled_coverage_cache'
COVER_DIR04.mkdir(parents=True,exist_ok=True)
UNIT_GROUP04=UNIT_CANDIDATES.loc[
    UNIT_CANDIDATES.common_bbox_area_m2.gt(0)].copy()
assert not UNIT_GROUP04.empty
def original_geom04(kind,uid):
    if kind=='plot':
        x=PLOTS[PLOTS.Plot_name.astype(str).eq(str(uid))]
    else:
        x=CIRCLES[CIRCLES.circle_id.astype(str).eq(str(uid))]
    assert len(x)==1,(kind,uid,len(x))
    return x.geometry.iloc[0]
def points04(geom):
    # Deterministic and concentrated on actual polygon interior.
    bounds=geom.bounds
    x0,y0,x1,y1=bounds
    coords=[geom.representative_point()]
    for yy in np.linspace(y0,y1,5)[1:-1]:
        for xx in np.linspace(x0,x1,5)[1:-1]:
            p=Point(float(xx),float(yy))
            if geom.covers(p):coords.append(p)
    for yy in np.linspace(y0,y1,7)[1:-1:2]:
        for xx in np.linspace(x0,x1,7)[1:-1:2]:
            p=Point(float(xx),float(yy))
            if geom.covers(p):coords.append(p)
    seen=set();out=[]
    for p in coords:
        key=(round(p.x,3),round(p.y,3))
        if key not in seen:
            out.append(p);seen.add(key)
        if len(out)>=MAX_POINTS04:break
    return out
def stamp04(paths):
    return [[str(p),int(p.stat().st_size),int(p.stat().st_mtime_ns)]
            for p in sorted(set(map(Path,paths)),key=str)]
def sample_row04(rec):
    mosaic=rec.mosaic
    geometry=original_geom04(rec.kind,rec.unit_id)
    groups=[k for k,v in MOSAIC_KEY.items() if v==mosaic]
    files=[RGB_DIR/mosaic,DSM_BY_RGB[mosaic],DATA/'Ortho_RGB/ROI.shp']
    for g in groups:files.extend(MS_BANDS[g].values())
    payload=dict(recipe=SAMPLE_ALGO04,kind=rec.kind,uid=rec.unit_id,
       site=rec.site,fold=int(rec.fold),mosaic=mosaic,
       polygon=geometry.wkb_hex,files=stamp04(files))
    fingerprint=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
    target=COVER_DIR04/(fingerprint[:24]+'.json')
    if target.is_file():
        old=json.loads(target.read_text())
        if old['fingerprint']==fingerprint:return old['result']
    sampled=0;valid_rgb=0;valid_dsm=0;valid_ms=0;valid_all=0
    byflight=defaultdict(int)
    with rasterio.Env(GDAL_CACHEMAX=32):
        with ExitStack() as stack:
            rgb=stack.enter_context(rasterio.open(RGB_DIR/mosaic))
            dsm_src=stack.enter_context(rasterio.open(DSM_BY_RGB[mosaic]))
            dsm=stack.enter_context(WarpedVRT(dsm_src,crs=rgb.crs,transform=rgb.transform,
                width=rgb.width,height=rgb.height,resampling=Resampling.bilinear))
            bands={}
            for group in groups:
                bands[group]={}
                for band in MS4:
                    src=stack.enter_context(rasterio.open(MS_BANDS[group][band]))
                    vrt=stack.enter_context(WarpedVRT(src,crs=rgb.crs,
                       transform=rgb.transform,width=rgb.width,height=rgb.height,
                       resampling=Resampling.bilinear,dtype='float32'))
                    ix=(MS4.index(band)+1) if group=='Baego_MS' else 1
                    bands[group][band]=(vrt,ix)
            seen=set()
            for p in points04(geometry):
                row,col=rgb.index(p.x,p.y)
                row=max(0,min(rgb.height-PATCH04,int(row)-PATCH04//2))
                col=max(0,min(rgb.width-PATCH04,int(col)-PATCH04//2))
                if (row,col) in seen:continue
                seen.add((row,col))
                win=Window(col,row,PATCH04,PATCH04)
                mask=rasterize([(geometry,1)],out_shape=(PATCH04,PATCH04),
                    transform=rgb.window_transform(win),dtype='uint8').astype(bool)
                if not mask.any():continue
                sampled+=int(mask.sum())
                rgb_mask=rgb.dataset_mask(window=win)>0
                rgb_arr=rgb.read([1,2,3],window=win,masked=True)
                rgb_mask &=~np.ma.getmaskarray(rgb_arr).any(axis=0)
                dsm_arr=dsm.read(1,window=win,masked=True,out_dtype='float32')
                dz=np.asarray(dsm_arr.filled(np.nan),np.float32)
                dsm_mask=(~np.ma.getmaskarray(dsm_arr))&np.isfinite(dz)
                accepted=[]
                for group,ff in bands.items():
                    bandvalid=np.ones((PATCH04,PATCH04),bool)
                    for band,(vrt,ix) in ff.items():
                        arr=vrt.read(ix,window=win,masked=True,out_dtype='float32')
                        z=np.asarray(arr.filled(0),np.float32)
                        bandvalid &= (~np.ma.getmaskarray(arr))&np.isfinite(z)&(z>0)
                    byflight[group]+=int((mask&rgb_mask&bandvalid).sum())
                    accepted.append(bandvalid)
                ms_mask=np.logical_or.reduce(accepted) if accepted else np.zeros(mask.shape,bool)
                valid_rgb+=int((mask&rgb_mask).sum())
                valid_dsm+=int((mask&rgb_mask&dsm_mask).sum())
                valid_ms+=int((mask&rgb_mask&ms_mask).sum())
                valid_all+=int((mask&rgb_mask&dsm_mask&ms_mask).sum())
    result=dict(kind=rec.kind,unit_id=str(rec.unit_id),site=rec.site,
        fold=int(rec.fold),mosaic=mosaic,n_sampled_polygon_px=sampled,
        n_rgb=valid_rgb,n_rgb_dsm=valid_dsm,n_rgb_ms=valid_ms,
        n_common=valid_all,estimated_common_fraction=valid_all/max(valid_rgb,1),
        acquisition_valid_sampled_px=json.dumps(byflight,sort_keys=True))
    tmp=target.with_suffix('.partial')
    tmp.write_text(json.dumps(dict(fingerprint=fingerprint,result=result),indent=2))
    os.replace(tmp,target)
    return result
ROWS04=[]
for j,rec in enumerate(UNIT_GROUP04.itertuples(index=False)):
    q=sample_row04(rec);ROWS04.append(q)
    if (j+1)%10==0 or j+1==len(UNIT_GROUP04):
        print('ACTUAL PIXEL QA',j+1,'/',len(UNIT_GROUP04),'last',q['site'],
              q['kind'],q['unit_id'],'common sampled',q['n_common'],flush=True)
PIXEL_QA04=pd.DataFrame(ROWS04)
PIXEL_QA04.to_csv(OUT/'05_sampled_valid_pixel_support_by_mosaic.csv',index=False)
display(PIXEL_QA04.groupby(['site','fold','kind']).agg(
  mosaic_unit_records=('unit_id','size'),
  n_with_common=('n_common',lambda x:int((x>0).sum())),
  sampled_common_frac=('estimated_common_fraction','median')).round(3))
print('NOTE: sampled coverage only. Exact support and registration still pending.')


## 05 — Deduplicated unit-level comparison eligibility


In [ ]:
# 05 — Unit-level deduplicated eligibility report and protocol provenance.
# Never count a unit twice just because two RGB mosaics/acquisitions overlap.
if 'PIXEL_QA04' not in globals():
    PIXEL_QA04=pd.read_csv(OUT/'05_sampled_valid_pixel_support_by_mosaic.csv')
p=PIXEL_QA04.copy()
p['unit_key']=p.kind.astype(str)+':'+p.unit_id.astype(str)
# Units may occur on multiple mosaics. Summarize with 'any actual sampled common support'.
eligible=p.groupby(['site','fold','kind','unit_key'],as_index=False).agg(
    n_mosaics=('mosaic','nunique'),
    n_sampled_sum=('n_sampled_polygon_px','sum'),
    n_common_sampled_sum=('n_common','sum'),
    highest_mosaic_coverage=('estimated_common_fraction','max'))
eligible['sampled_eligible']=eligible.n_common_sampled_sum.gt(0)
eligible.to_csv(OUT/'05_deduplicated_unit_candidate_eligibility.csv',index=False)
print('DEDUPLICATED ORIGINAL UNITS (never pixels or mosaic records as independent replicates):')
display(eligible.groupby(['site','fold','kind']).agg(
  n_unique_units=('unit_key','nunique'),
  n_with_sampled_common=('sampled_eligible','sum'),
  median_max_sampled_coverage=('highest_mosaic_coverage','median')).round(3))
print('YOKO CANDIDATE UNITS (band 1 in both P1 and P2):')
display(eligible.loc[eligible.site.eq('Yoko')])
print('WARNING: zero sampled overlap can miss narrow strips; no units are permanently excluded')
print('until exact pixel-validity coverage and RGB–MS coregistration are audited.')
import json
protocol_path=OUT/'05_optionB_frozen_protocol_STAGE1.json'
assert protocol_path.is_file(),'Run 03 first'
protocol=json.loads(protocol_path.read_text())
protocol['stage1_sampled_pixel_support']=str(OUT/'05_sampled_valid_pixel_support_by_mosaic.csv')
protocol['stage1_deduplicated_unit_screen']=str(OUT/'05_deduplicated_unit_candidate_eligibility.csv')
protocol['status']='STAGE1_SAMPLED_PIXEL_QA_COMPLETE_NOT_EXACT_SUPPORT_NOT_REGISTERED'
protocol_path.write_text(json.dumps(protocol,indent=2))
print('READY TO REVIEW, NOT READY TO TRAIN:',protocol_path)


## 06 — Exact full-reference-unit pixel coverage (bounded windows)

Use every frozen field plot and eligible complete circle, **even where no MS bounding-box overlap exists**. RGB, RGB+DSM, RGB+MS, and the RGB+DSM+MS triple intersection are counted against the same original ROI-clipped RGB-valid unit. A four-band MS pixel is valid only if red, green, red-edge and NIR are present **within the same acquisition**. The P1 and P2 union does not mix their bands.

This is a potentially substantial windowed read of existing rasters, but no model training. Each unit–mosaic is cached atomically for resumability. A unit intersecting multiple RGB mosaics is represented in the primary report by the mosaic with the largest verified common-support count; the full set of mosaic records is retained for audit, and no double counting is permitted.


In [ ]:
# 06 — Full polygon, exact raster-valid pixel-support audit, no model inference.
# Run 00 -> 01 -> 06 -> 07. Uses bounded 512-pixel windows and resumes by unit/mosaic.
import json,hashlib,os,time
from contextlib import ExitStack
from collections import defaultdict
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from shapely.geometry import box
from shapely.strtree import STRtree
from rasterio.warp import transform_bounds
BLOCK06=512
REPORT06=OUT/'06_exact_full_unit_support'
CACHE06=REPORT06/'unit_mosaic_cache'
CACHE06.mkdir(parents=True,exist_ok=True)
ALL_GROUPS06=defaultdict(list)
for group,name in MOSAIC_KEY.items():ALL_GROUPS06[name].append(group)
DSM_BY_RGB06={Path(z.rgb_path).name:Path(z.dsm_path) for z in PAIRS.itertuples(index=False)}
RGB_MOSAICS06=sorted(DSM_BY_RGB06)
# Preflight DSM TIFF header integrity before processing any expensive units.
# A TIFFReadDirectory error is an unreadable source file, not empty DSM coverage.
DSM_OPEN_ERRORS06=[]
for mosaic_check, dsm_path in DSM_BY_RGB06.items():
    try:
        with rasterio.Env(GDAL_CACHEMAX=16):
            with rasterio.open(dsm_path) as ds_check:
                assert ds_check.count>=1 and ds_check.crs is not None
                # Windowed read checks both metadata and at least one tile.
                probe=Window(0,0,min(16,ds_check.width),min(16,ds_check.height))
                ds_check.read(1,window=probe,masked=True)
        print('DSM HEADER/PROBE OK:',mosaic_check,Path(dsm_path).name,flush=True)
    except Exception as exc:
        DSM_OPEN_ERRORS06.append((mosaic_check,str(dsm_path),repr(exc)))
if DSM_OPEN_ERRORS06:
    print('UNREADABLE DSM SOURCE FILE(S):',flush=True)
    for mosaic_check,dsm_path,error in DSM_OPEN_ERRORS06:
        p=Path(dsm_path)
        print(' ',mosaic_check,'file',dsm_path,
              'bytes',p.stat().st_size if p.is_file() else 'NOT PRESENT',
              'error',error,flush=True)
    raise RuntimeError(
        'DSM TIFF preflight failed. A failed TIFFReadDirectory indicates a file-level '
        'read error (possibly incomplete/corrupt TIFF or transient Drive mount). '
        'No scientific coverage estimate should treat this as empty DSM. '
        'First test rasterio.open on the same DSM in a new Drive session; if still '
        'unreadable, replace/re-export that TIFF from a verified source and rerun 06. '
        'Already completed unit caches remain untouched.')
assert RGB_MOSAICS06 and set(ALL_GROUPS06).issubset(set(RGB_MOSAICS06))
ROI_GEOMS06=list(ROI.geometry)
ROI_TREE06=STRtree(ROI_GEOMS06)
BAND_LIST06=('red','green','red_edge','nir')
def burn06(polys,ds,window):
    sh=(int(window.height),int(window.width))
    shapes=[(g,1) for g in polys if g is not None and not g.is_empty]
    if not shapes:return np.zeros(sh,bool)
    return rasterize(shapes,out_shape=sh,transform=ds.window_transform(window),
                     fill=0,dtype='uint8').astype(bool)
def vrt_group06(ds,group,stack):
    bands={}
    for band in BAND_LIST06:
        p=MS_BANDS[group][band]
        sr=stack.enter_context(rasterio.open(p))
        ix=BAND_LIST06.index(band)+1 if group=='Baego_MS' else 1
        assert sr.count>=ix
        vrt=stack.enter_context(WarpedVRT(sr,crs=ds.crs,transform=ds.transform,
               width=ds.width,height=ds.height,resampling=Resampling.bilinear,
               dtype='float32'))
        bands[band]=(vrt,ix)
    return bands
def signal_mask06(vrt,ix,win):
    v=vrt.read(ix,window=win,masked=True,out_dtype='float32')
    a=np.asarray(v.filled(0),dtype=np.float32)
    return (~np.ma.getmaskarray(v))&np.isfinite(a)&(a>0)
def file_stamp06(paths):
    return [[str(p),p.stat().st_size,p.stat().st_mtime_ns]
        for p in sorted(set(map(Path,paths)),key=str)]
def calculate06(unit,mosaic,ds,dsm,groups):
    uid=str(unit['uid']);kind=unit['kind'];geom=unit['geometry']
    input_paths=[Path(ds.name),DSM_BY_RGB06[mosaic],
                 DATA/'Ortho_RGB/ROI.shp',Path(CONTRACT['folds_csv']),
                 REF/'01_field78_circle_folds.csv']
    for g in groups:input_paths.extend(MS_BANDS[g].values())
    signature=hashlib.sha256(json.dumps(dict(
        recipe='NB05_V4_003_full_unit_ROI_RGB_DSM_fourband_perflight_v1',
        uid=uid,kind=kind,fold=int(unit['fold']),mosaic=mosaic,
        geometry=geom.wkb_hex,source=file_stamp06(input_paths)),sort_keys=True).encode()).hexdigest()
    target=CACHE06/(signature[:32]+'.json')
    if target.is_file():
        q=json.loads(target.read_text())
        if q.get('signature')==signature:return q['result']
    raw=rasterio.windows.from_bounds(*geom.bounds,transform=ds.transform)
    r0=max(0,int(np.floor(raw.row_off)))
    c0=max(0,int(np.floor(raw.col_off)))
    r1=min(ds.height,int(np.ceil(raw.row_off+raw.height)))
    c1=min(ds.width,int(np.ceil(raw.col_off+raw.width)))
    counts=dict(original_unit_rgb_px=0,rgb_dsm_px=0,
                rgb_fourband_ms_px=0,common_rgb_dsm_ms_px=0)
    counts_by_group={g:0 for g in groups}
    for rr in range(r0,r1,BLOCK06):
        for cc in range(c0,c1,BLOCK06):
            win=Window(cc,rr,min(BLOCK06,r1-rr),min(BLOCK06,c1-cc))
            shape=(int(win.height),int(win.width))
            polygon=burn06([geom],ds,win)
            if not polygon.any():continue
            bb=box(*rasterio.windows.bounds(win,ds.transform))
            roi_hits=[ROI_GEOMS06[int(i)] for i in
                ROI_TREE06.query(bb,predicate='intersects')]
            inside=polygon&burn06(roi_hits,ds,win)
            if not inside.any():continue
            rgbarr=ds.read([1,2,3],window=win,masked=True)
            rgb_valid=inside&(ds.dataset_mask(window=win)>0)
            rgb_valid &= ~np.ma.getmaskarray(rgbarr).any(axis=0)
            if not rgb_valid.any():continue
            dz=dsm.read(1,window=win,masked=True,out_dtype='float32')
            dsm_valid=(~np.ma.getmaskarray(dz))&np.isfinite(
                np.asarray(dz.filled(np.nan),np.float32))
            ms_union=np.zeros(shape,bool)
            for group,bandmap in groups.items():
                one=np.ones(shape,bool)
                for band in BAND_LIST06:
                    vrt,ix=bandmap[band]
                    one&=signal_mask06(vrt,ix,win)
                covered=one&rgb_valid
                counts_by_group[group]+=int(covered.sum())
                ms_union|=one
            counts['original_unit_rgb_px']+=int(rgb_valid.sum())
            counts['rgb_dsm_px']+=int((rgb_valid&dsm_valid).sum())
            counts['rgb_fourband_ms_px']+=int((rgb_valid&ms_union).sum())
            counts['common_rgb_dsm_ms_px']+=int(
                (rgb_valid&dsm_valid&ms_union).sum())
    denominator=counts['original_unit_rgb_px']
    result=dict(site=unit['site'],fold=int(unit['fold']),kind=kind,
        unit_id=uid,mosaic=mosaic,**counts,
        ms_pct_of_rgb=100*counts['rgb_fourband_ms_px']/denominator if denominator else np.nan,
        dsm_pct_of_rgb=100*counts['rgb_dsm_px']/denominator if denominator else np.nan,
        common_pct_of_rgb=100*counts['common_rgb_dsm_ms_px']/denominator if denominator else np.nan,
        acquisition_covered_rgb_pixels=json.dumps(counts_by_group,sort_keys=True))
    tmp=target.with_suffix('.partial')
    tmp.write_text(json.dumps(dict(signature=signature,result=result),indent=2))
    os.replace(tmp,target)
    return result
UNITS06=[]
for q in PLOTS.itertuples(index=False):
    UNITS06.append(dict(site=str(q.site_key),fold=int(q.cv_fold),kind='plot',
        uid=str(q.Plot_name),geometry=q.geometry))
for q in CIRCLES.itertuples(index=False):
    if bool(q.cross_fold_plot_intersection):continue
    UNITS06.append(dict(site=str(q.site_key),fold=int(q.cv_fold),kind='circle',
        uid=str(q.circle_id),geometry=q.geometry))
assert len([q for q in UNITS06 if q['kind']=='plot'])==78
ROWS06=[]
for mosaic in RGB_MOSAICS06:
    candidates=[]
    with rasterio.open(RGB_DIR/mosaic) as ds:
        assert ds.crs is not None
        full=box(*ds.bounds)
        candidates=[q for q in UNITS06 if q['geometry'].intersects(full)
            and bool(q['geometry'].intersection(full).area>0) and
            q['site']==RGB_TO_SITE[mosaic]]
        print('MOSAIC',mosaic,'candidate full original units',len(candidates),flush=True)
        if not candidates:continue
        with rasterio.Env(GDAL_CACHEMAX=64):
            with ExitStack() as stack:
                dsm_src=stack.enter_context(rasterio.open(DSM_BY_RGB06[mosaic]))
                dsm=stack.enter_context(WarpedVRT(dsm_src,crs=ds.crs,
                    transform=ds.transform,width=ds.width,height=ds.height,
                    resampling=Resampling.bilinear,dtype='float32'))
                group_vrts={g:vrt_group06(ds,g,stack)
                            for g in ALL_GROUPS06.get(mosaic,[])}
                for j,u in enumerate(candidates):
                    rec=calculate06(u,mosaic,ds,dsm,group_vrts)
                    ROWS06.append(rec)
                    if (j+1)%5==0 or (j+1)==len(candidates):
                        print('FULL-SUPPORT',mosaic,j+1,'/',len(candidates),
                              rec['kind'],rec['unit_id'],
                              'common=',rec['common_rgb_dsm_ms_px'],
                              'pct=',round(rec['common_pct_of_rgb'],1),flush=True)
    pd.DataFrame(ROWS06).to_csv(REPORT06/'06_full_unit_mosaic_partial.csv',index=False)
EXACT06=pd.DataFrame(ROWS06)
assert not EXACT06.empty
EXACT06.to_csv(REPORT06/'06_full_unit_mosaic_support.csv',index=False)
print('FULL-UNIT SENSOR VALIDITY AUDIT COMPLETE',len(EXACT06),
      'unit–mosaic records (before deduplication)')
display(EXACT06.groupby(['site','fold','kind']).agg(
    n_unit_mosaic=('unit_id','size'),
    with_common=('common_rgb_dsm_ms_px',lambda x:int((x>0).sum())),
    median_common_pct=('common_pct_of_rgb','median')).round(2))


## 07 — Deduplicated full-unit evaluation support including combined Yoko coverage


In [ ]:
# 07 — Keep all 78 original plots and eligible circles, including zero-support units.
if 'EXACT06' not in globals():
    EXACT06=pd.read_csv(REPORT06/'06_full_unit_mosaic_support.csv')
D=EXACT06.copy()
D['unit_key']=D.kind.astype(str)+':'+D.unit_id.astype(str)
D=D.sort_values(['common_rgb_dsm_ms_px','original_unit_rgb_px','mosaic'],
    ascending=[False,False,True])
# Coverage-only mosaic choice, independent of segmentation truth or predictions.
SELECTED07=D.drop_duplicates(['unit_key'],keep='first').copy()
all_unit_keys={q['kind']+':'+q['uid'] for q in UNITS06}
missing=all_unit_keys-set(SELECTED07.unit_key)
for k in sorted(missing):
    u=next(x for x in UNITS06 if x['kind']+':'+x['uid']==k)
    SELECTED07=pd.concat([SELECTED07,pd.DataFrame([dict(
        site=u['site'],fold=u['fold'],kind=u['kind'],unit_id=u['uid'],
        mosaic='NO_RGB_INTERSECTION',unit_key=k,original_unit_rgb_px=0,
        rgb_dsm_px=0,rgb_fourband_ms_px=0,common_rgb_dsm_ms_px=0,
        ms_pct_of_rgb=np.nan,dsm_pct_of_rgb=np.nan,common_pct_of_rgb=np.nan,
        acquisition_covered_rgb_pixels='{}')])],ignore_index=True)
assert set(SELECTED07.unit_key)==all_unit_keys
assert not SELECTED07.unit_key.duplicated().any()
SELECTED07['eligible_common']=SELECTED07.common_rgb_dsm_ms_px.gt(0)
SELECTED07.to_csv(REPORT06/'07_frozen_deduplicated_original_unit_support.csv',index=False)
MATRIX07=SELECTED07.groupby(['site','fold','kind']).agg(
    n_original_units=('unit_key','size'),
    n_common=('eligible_common','sum'),
    median_common_pct=('common_pct_of_rgb','median')).round(2)
display(MATRIX07)
print('YOKO: combined four-band MS coverage (union of P1 and P2) by original reference unit')
display(SELECTED07.loc[SELECTED07.site.eq('Yoko'),[
    'fold','kind','unit_id','mosaic','original_unit_rgb_px',
    'rgb_dsm_px','rgb_fourband_ms_px','common_rgb_dsm_ms_px',
    'common_pct_of_rgb']])
print('ALL-UNIT COUNT',len(SELECTED07),'of',len(UNITS06),
      'common supported',int(SELECTED07.eligible_common.sum()))
print('WARNINGS: alignment is not yet verified; effective DSM-context support')
print('with multiscale filter halos may be smaller. No training has been run.')
protocol_file=OUT/'05_optionB_frozen_protocol_STAGE1.json'
if protocol_file.is_file():
    protocol=json.loads(protocol_file.read_text())
else:
    protocol={'name':'NB05 Option B harmonized modality comparison'}
protocol.update(status='STAGE1_EXACT_PIXEL_SUPPORT_READY_REGISTRATION_PENDING',
    exact_unit_mosaic_support=str(REPORT06/'06_full_unit_mosaic_support.csv'),
    frozen_deduplicated_support=str(
        REPORT06/'07_frozen_deduplicated_original_unit_support.csv'))
protocol_file.write_text(json.dumps(protocol,indent=2))
print('UPDATED CONTRACT',protocol_file)


## 08 — Common RGB + DSM + MS chip cache (resumable)

This is the **only new sensor extraction** required for harmonized training. Each fixed outer fold reads the immutable NB02 labelled training chips and NB03 spatial inner-role audit; no outer validation labels are consulted. Each chip receives the same ROI/source mask, DSM structural predictors and relative four-band MS spectra. The training dataset is the RGB–DSM–MS **joint-valid intersection**. Each mosaic–fold cache is atomic and fingerprinted. A GPU is *not* required for this step, but keep a GPU runtime if you intend to continue into training in the same session.


In [ ]:
# 08 — Preflight frozen source labels, folds, and exact DSM feature definition.
import os,json,hashlib,sys,subprocess,importlib.util,gc,copy,time
from pathlib import Path
from collections import defaultdict
from contextlib import ExitStack
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
import torch
from scipy.ndimage import uniform_filter
if importlib.util.find_spec('segmentation_models_pytorch') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','segmentation-models-pytorch'])
import segmentation_models_pytorch as smp
CHIP08=256
FOLDS08=(0,1,2)
SOURCE_RGB08=ROOT/'Outputs/v4/02_rgb_unet_field78_v4_003_full/chip_cache'
ALT_RGB08=ROOT/'Outputs/v4/03_rgb_dsm_multiscale_v4_007/chip_cache'
INNER08=ROOT/'Outputs/v4/03_rgb_dsm_multiscale_v4_007'
BASE08=ROOT/'Outputs/v4/02_rgb_unet_field78_v4_007_full'
RGB_CKPT08={f:BASE08/f'NB02_V4_007_fold{f}_best.pt' for f in FOLDS08}
assert all(p.is_file() for p in RGB_CKPT08.values())
DSM08={Path(z.rgb_path).name:Path(z.dsm_path) for z in PAIRS.itertuples(index=False)}
GROUPS08=defaultdict(list)
for g,m in MOSAIC_KEY.items():GROUPS08[m].append(g)
CACHE08=ROOT/'Outputs/v4/05_optionB_joint_chips_cache_v1'
CACHE08.mkdir(parents=True,exist_ok=True)
TRAINING08=OUT/'08_joint_chip_manifest.csv'
PROMINENCE_RADIUS_M=6.
ROUGHNESS_RADIUS_M=3.
CONVEXITY_RADIUS_M=2.
CONTEXT_RESOLUTION_M=1.0
CONTEXT_MAX_RADIUS_M=60.0

# 02 — Windowed, halo-safe structural feature extraction
from contextlib import ExitStack
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from scipy.ndimage import uniform_filter
def _mean_valid(a,valid,size):
    f=valid.astype(np.float32)
    num=uniform_filter(np.where(valid,a,0.).astype(np.float32),size=size,mode='nearest')
    den=uniform_filter(f,size=size,mode='nearest')
    return num/np.maximum(den,1e-6),den

def _odd_pixels(radius_m,pixel_m):
    r=max(1,int(np.ceil(radius_m/pixel_m)))
    return 2*r+1

def local_feature_tile(rgb_path,dsm_path,win):
    """Return 6xHxW [RGB3, prominence, roughness, convexity], joint valid HxW.
    Requires actual pixel registration to be separately approved before training.
    """
    with ExitStack() as stack:
        # Accept already-open raster handles to avoid hundreds of repeated GDAL opens.
        # Original path-based calls (including held-out inference) still work.
        rgb=(rgb_path if hasattr(rgb_path,'read') else stack.enter_context(rasterio.open(rgb_path)))
        dsm=(dsm_path if hasattr(dsm_path,'read') else stack.enter_context(rasterio.open(dsm_path)))
        assert rgb.crs is not None and dsm.crs is not None
        h,w=int(win.height),int(win.width)
        assert h>0 and w>0 and int(win.row_off)>=0 and int(win.col_off)>=0
        assert win.row_off+h<=rgb.height and win.col_off+w<=rgb.width
        step=min(abs(rgb.transform.a),abs(rgb.transform.e))
        max_radius=max(PROMINENCE_RADIUS_M,ROUGHNESS_RADIUS_M,
                       CONVEXITY_RADIUS_M)
        halo=int(np.ceil(2.0*max_radius/step))+4
        r0=max(0,int(win.row_off)-halo);c0=max(0,int(win.col_off)-halo)
        r1=min(rgb.height,int(win.row_off)+h+halo)
        c1=min(rgb.width,int(win.col_off)+w+halo)
        expanded=Window(c0,r0,c1-c0,r1-r0)
        tr=rgb.window_transform(expanded)
        rgb_raw=rgb.read([1,2,3],window=expanded,masked=True)
        if not all(np.dtype(t)==np.dtype('uint8') for t in rgb.dtypes[:3]):
            raise ValueError(f'RGB is not uint8: {rgb_path}: {rgb.dtypes}')
        rgb_valid=(rgb.dataset_mask(window=expanded)>0)&(
            ~np.ma.getmaskarray(rgb_raw).any(axis=0))
        rgb_data=np.asarray(rgb_raw.filled(0),dtype=np.float32)/255.
        vrt=stack.enter_context(WarpedVRT(dsm,crs=rgb.crs,transform=tr,
            width=int(expanded.width),height=int(expanded.height),
            resampling=Resampling.bilinear,src_nodata=dsm.nodata,
            nodata=np.nan))
        arr=vrt.read(1,masked=True)
        z=np.asarray(arr.data,dtype=np.float32).copy()
        dsm_valid=(~np.ma.getmaskarray(arr))&np.isfinite(z)
        valid=rgb_valid&dsm_valid
        z0=np.where(dsm_valid,z,0.).astype(np.float32)
        prominence_size=_odd_pixels(PROMINENCE_RADIUS_M,step)
        rough_size=_odd_pixels(ROUGHNESS_RADIUS_M,step)
        convex_size=_odd_pixels(CONVEXITY_RADIUS_M,step)
        base,base_support=_mean_valid(z0,dsm_valid,prominence_size)
        local,rough_support=_mean_valid(z0,dsm_valid,rough_size)
        local2,_=_mean_valid(z0*z0,dsm_valid,rough_size)
        smooth,conv_support=_mean_valid(z0,dsm_valid,convex_size)
        prominence=z0-base
        roughness=np.sqrt(np.maximum(0.,local2-local**2))
        convexity=z0-smooth
        # Exclude filters with inadequate valid support near DSM boundaries.
        valid &= (base_support>=.95)&(rough_support>=.95)&(conv_support>=.95)
        y=int(win.row_off)-r0;x=int(win.col_off)-c0
        sl=np.s_[y:y+h,x:x+w]
        x6=np.concatenate([rgb_data[:,sl[0],sl[1]],
            prominence[sl][None],roughness[sl][None],
            convexity[sl][None]],axis=0).astype(np.float32)
        joint=valid[sl].copy()
        x6[:,~joint]=0.
        assert x6.shape==(6,h,w) and joint.shape==(h,w)
        assert np.isfinite(x6).all()
        return x6,joint

# 1 m global UTM-anchored context grid: consistent regardless of chip positions.
from rasterio.warp import reproject
from rasterio.transform import Affine
from scipy.ndimage import uniform_filter
from math import floor,ceil

CONTEXT_BANDS=('relative_surface_5_15m','relative_surface_15_30m',
    'relative_surface_30_60m','surface_sd_15m','surface_sd_30m',
    'surface_sd_60m','context_valid_fraction_30_60m')
FEATURE_NAMES=('R','G','B','local_prominence_m','surface_roughness_m',
               'local_convexity_m')+CONTEXT_BANDS
N_DSM_FEATURES=len(FEATURE_NAMES)-3
assert N_DSM_FEATURES==10  # 3 local + 7 contextual
def _box_mean(data,valid,r):
    size=2*int(r)+1
    mask=valid.astype(np.float32)
    val=np.where(valid,data,0.).astype(np.float32)
    count=uniform_filter(mask,size=size,mode='constant',cval=0.)
    total=uniform_filter(val,size=size,mode='constant',cval=0.)
    return total/np.maximum(count,1e-6),count

def _annular_mean(data,valid,inner,outer):
    # uniform_filter returns average over a square window; reconstruct
    # weighted sums for annular mean, excluding its inner square.
    def sums(radius):
        size=2*radius+1
        cnt=uniform_filter(valid.astype(np.float32),size=size,mode='constant',cval=0.)*(size*size)
        val=uniform_filter(np.where(valid,data,0.).astype(np.float32),size=size,
                           mode='constant',cval=0.)*(size*size)
        return val,cnt
    so,no=sums(int(outer))
    si,ni=sums(int(inner))
    count=np.maximum(no-ni,0.)
    return (so-si)/np.maximum(count,1e-6),np.clip(count/max(1,(2*outer+1)**2-(2*inner+1)**2),0.,1.)

def context_feature_tile(rgb,dsm,win):
    """Seven spatial contextual features at native RGB grid, coarse computation at 1 m."""
    bounds=rasterio.windows.bounds(win,rgb.transform)
    xmin,ymin,xmax,ymax=bounds
    pad=CONTEXT_MAX_RADIUS_M+4*CONTEXT_RESOLUTION_M
    unit=CONTEXT_RESOLUTION_M
    left=floor((xmin-pad)/unit)*unit
    right=ceil((xmax+pad)/unit)*unit
    top=ceil((ymax+pad)/unit)*unit
    bottom=floor((ymin-pad)/unit)*unit
    w=int(round((right-left)/unit));h=int(round((top-bottom)/unit))
    assert 0<w<=8192 and 0<h<=8192,(h,w,'Unexpected context window')
    tr=Affine(unit,0,left,0,-unit,top)
    with WarpedVRT(dsm,crs=rgb.crs,transform=tr,width=w,height=h,
                   resampling=Resampling.bilinear,src_nodata=dsm.nodata,nodata=np.nan) as vrt:
        raster=vrt.read(1,masked=True)
    z=np.asarray(raster.data,dtype=np.float32)
    valid=(~np.ma.getmaskarray(raster))&np.isfinite(z)
    # Preserve relative heights but remove global offset before variance estimation.
    reference=float(np.median(z[valid])) if np.any(valid) else 0.
    relative=np.where(valid,z-reference,0.).astype(np.float32)
    local,_=_box_mean(relative,valid,3)
    surfaces=[];fractions=[]
    for inner,outer in ((5,15),(15,30),(30,60)):
        surround,frac=_annular_mean(relative,valid,inner,outer)
        surfaces.append(np.where(frac>=.6,local-surround,0.).astype(np.float32))
        fractions.append(frac)
    rough=[]
    for radius in (15,30,60):
        mean,cov=_box_mean(relative,valid,radius)
        second,_=_box_mean(relative*relative,valid,radius)
        rough.append(np.where(cov>=.6,np.sqrt(np.maximum(second-mean*mean,0.)),0.).astype(np.float32))
    coarse=np.stack(surfaces+rough+[fractions[2]],axis=0)
    dst_transform=rgb.window_transform(win)
    out=np.zeros((len(CONTEXT_BANDS),int(win.height),int(win.width)),dtype=np.float32)
    # Smooth resampling of contextual maps. No extrapolation beyond coarse coverage.
    for i in range(len(CONTEXT_BANDS)):
        reproject(source=coarse[i],destination=out[i],
                  src_transform=tr,src_crs=rgb.crs,dst_transform=dst_transform,
                  dst_crs=rgb.crs,resampling=Resampling.bilinear,
                  src_nodata=None,dst_nodata=-9999.)
    # 0 is a legitimate relative-elevation/roughness value. It must NOT be
    # used as destination NoData: GDAL may perturb valid zeros to 1.4013e-45.
    # Convert only pixels explicitly initialized to -9999 back to 0 in the
    # final feature tensor (the contextual coverage band carries availability).
    out[out==-9999.]=0.
    assert np.isfinite(out).all()
    return out

def feature_tile(rgb_path,dsm_path,win):
    # Handles may be passed directly by the 25-chip persistent cache.
    with ExitStack() as stack:
        rgb=rgb_path if hasattr(rgb_path,'read') else stack.enter_context(rasterio.open(rgb_path))
        dsm=dsm_path if hasattr(dsm_path,'read') else stack.enter_context(rasterio.open(dsm_path))
        local,joint=local_feature_tile(rgb,dsm,win)
        ctx=context_feature_tile(rgb,dsm,win)
    out=np.concatenate([local,ctx],axis=0)
    out[:,~joint]=0.
    assert out.shape==(3+N_DSM_FEATURES,int(win.height),int(win.width))
    assert np.isfinite(out).all()
    return out,joint

print('Multiscale DSM features:',FEATURE_NAMES)


### 08A — Extract and resume fold/mosaic caches

This cell writes a compact cache per fold and mosaic, and resumes completed files by content fingerprint. It reads only 256 × 256 source-chip windows (DSM contextual calculations extend to a 60 m neighborhood). Expect significant I/O from Google Drive, particularly for the large MOS TIFFs. Each completed mosaic is reusable.


In [ ]:
# 08A — Source-labelled cache -> joint valid RGB, DSM10, MS4 and identical labels.
# Strict outer-fold train-only + frozen spatially blocked inner selection.
import hashlib
from contextlib import ExitStack
from rasterio.vrt import WarpedVRT
def file_sig08(paths):
    return [[str(p),int(Path(p).stat().st_size),int(Path(p).stat().st_mtime_ns)]
            for p in sorted(set(map(Path,paths)),key=str)]
def rgb_source08(fold):
    a=sorted(SOURCE_RGB08.glob(f'field78_train_fold{fold}_*.npz'))
    if not a:a=sorted(ALT_RGB08.glob(f'field78_train_fold{fold}_*.npz'))
    assert len(a)==1,('Expected exactly one frozen RGB train cache',fold,a)
    return a[0]
def strict_role08(fold):
    f=INNER08/f'03_v4_002_inner_spatial_split_fold{fold}.csv'
    assert f.is_file(),('Original blocked inner-role audit missing',f)
    x=pd.read_csv(f)
    assert not x.duplicated(['mosaic','row','col']).any()
    return f,{(str(z.mosaic),int(z.row),int(z.col)):str(z.inner_role)
              for z in x.itertuples(index=False)}
def bands_vrt08(rgb,groups,stack):
    result={}
    for group in groups:
        result[group]={}
        for band in ('red','green','red_edge','nir'):
            src=stack.enter_context(rasterio.open(MS_BANDS[group][band]))
            idx=('red','green','red_edge','nir').index(band)+1 if group=='Baego_MS' else 1
            vrt=stack.enter_context(WarpedVRT(src,crs=rgb.crs,
                transform=rgb.transform,width=rgb.width,height=rgb.height,
                resampling=Resampling.bilinear,dtype='float32'))
            result[group][band]=(vrt,idx)
    return result
def matched_ms08(vrts,win):
    stack=[];supported=[]
    for band in ('red','green','red_edge','nir'):
        ds,idx=vrts[band]
        a=ds.read(idx,window=win,masked=True,out_dtype='float32')
        v=np.asarray(a.filled(0),np.float32)
        ok=(~np.ma.getmaskarray(a))&np.isfinite(v)&(v>0)
        stack.append(np.where(ok,v,0))
        supported.append(ok)
    raw=np.stack(stack)
    valid=np.logical_and.reduce(supported)
    total=raw.sum(axis=0)
    valid &=total>1e-6
    relative=np.where(valid[None],raw/np.maximum(total[None],1e-6),0)
    return relative.astype('float16'),valid
def build_fold08(fold):
    src=rgb_source08(fold)
    splitfile,roles=strict_role08(fold)
    with np.load(src,allow_pickle=False) as z:
        for col in ['rgb','y','w','source','site','mosaic','row','col']:
            assert col in z.files,(src,col)
        count=len(z['rgb'])
        assert z['rgb'].shape==(count,256,256,3)
        # Read only small arrays first; load source RGB chips once, sequentially by mosaic.
        names=z['mosaic'].astype(str)
        rows=z['row'].astype(int)
        cols=z['col'].astype(int)
        sites=z['site'].astype(str)
        bymosaic=defaultdict(list)
        for i in range(count):
            key=(names[i],int(rows[i]),int(cols[i]))
            role=roles.get(key)
            if role in ('inner_train','inner_val_candidate') and names[i] in GROUPS08:
                bymosaic[names[i]].append((i,role))
        assert bymosaic,('No eligible RGB-labelled training windows',fold)
        new_records=[]
        for mosaic,items in sorted(bymosaic.items()):
            groups=GROUPS08[mosaic]
            sources=[src,splitfile,DATA/'Ortho_RGB'/mosaic,DSM08[mosaic]]
            for g in groups:sources.extend(MS_BANDS[g].values())
            sig=hashlib.sha256(json.dumps(dict(
                 recipe='NB05_OptionB_V4_006_joint_MS4_DSM10_v1',
                 fold=fold,mosaic=mosaic,
                 indices=[int(q[0]) for q in items],files=file_sig08(sources)),
                 sort_keys=True).encode()).hexdigest()
            dest=CACHE08/f'fold{fold}_{hashlib.sha1(mosaic.encode()).hexdigest()[:10]}_{sig[:20]}.npz'
            if dest.is_file():
                try:
                    with np.load(dest,allow_pickle=False) as v:
                        ok=(str(v['signature'].item())==sig and
                            v['dsm'].shape[1:]==(10,256,256) and
                            v['ms'].shape[1:]==(4,256,256) and
                            v['rgb'].shape[1:]==(256,256,3))
                    if ok:
                        print('JOINT CHIP CACHE HIT',fold,mosaic,flush=True)
                        new_records.append(dict(fold=fold,mosaic=mosaic,
                            path=str(dest),signature=sig));continue
                except Exception as e:print('INVALID CACHE',dest,repr(e))
            print('EXTRACT JOINT CHIPS',fold,mosaic,len(items),flush=True)
            records=[]
            with rasterio.Env(GDAL_CACHEMAX=128):
                with ExitStack() as stack:
                    rgb=stack.enter_context(rasterio.open(DATA/'Ortho_RGB'/mosaic))
                    dsm=stack.enter_context(rasterio.open(DSM08[mosaic]))
                    vrts=bands_vrt08(rgb,groups,stack)
                    for j,(ix,role) in enumerate(items):
                        row,col=int(rows[ix]),int(cols[ix])
                        win=Window(col,row,256,256)
                        if row<0 or col<0 or row+256>rgb.height or col+256>rgb.width:continue
                        # No possible cross-flight mixing of spectral bands.
                        candidate=[matched_ms08(vrts[g],win) for g in groups]
                        ms4,msok=max(candidate,key=lambda v:int(v[1].sum()))
                        if not msok.any():continue
                        xyz,dsmok=feature_tile(rgb,dsm,win)
                        label=z['y'][ix].astype('uint8')
                        support=z['w'][ix].astype(bool)&msok&dsmok
                        source=z['source'][ix].astype('uint8')
                        # Inner selection: exhaustive complete plot/circle only.
                        if role=='inner_val_candidate':
                            support&=source>=2
                        if support.sum()<50:continue
                        source[~support]=0
                        records.append(dict(rgb=z['rgb'][ix],dsm=xyz[3:].astype('float16'),
                            ms=ms4,y=label,w=support.astype('uint8'),source=source,
                            site=str(sites[ix]),role=role,row=row,col=col))
                        if (j+1)%50==0:print(' ',fold,mosaic,j+1,'/',len(items),
                            'retained',len(records),flush=True)
            if not records:
                print('NO JOINT CHIPS',fold,mosaic,flush=True);continue
            tmp=dest.with_suffix('.partial.npz')
            np.savez_compressed(tmp,signature=np.array(sig),
               rgb=np.stack([q['rgb'] for q in records]).astype('uint8'),
               dsm=np.stack([q['dsm'] for q in records]).astype('float16'),
               ms=np.stack([q['ms'] for q in records]).astype('float16'),
               y=np.stack([q['y'] for q in records]).astype('uint8'),
               w=np.stack([q['w'] for q in records]).astype('uint8'),
               source=np.stack([q['source'] for q in records]).astype('uint8'),
               site=np.array([q['site'] for q in records],dtype='U32'),
               role=np.array([q['role'] for q in records],dtype='U32'),
               row=np.array([q['row'] for q in records],dtype='int32'),
               col=np.array([q['col'] for q in records],dtype='int32'))
            os.replace(tmp,dest)
            print('JOINT CHIPS SAVED',fold,mosaic,len(records),flush=True)
            new_records.append(dict(fold=fold,mosaic=mosaic,path=str(dest),signature=sig))
        return new_records
ROWS08=[]
for fold in FOLDS08:
    ROWS08.extend(build_fold08(fold))
    pd.DataFrame(ROWS08).to_csv(TRAINING08,index=False)
MANIFEST08=pd.DataFrame(ROWS08)
assert not MANIFEST08.empty
MANIFEST08.to_csv(TRAINING08,index=False)
display(MANIFEST08)
print('ALL COMMON-SUPPORT TRAINING CHIPS READY; no outer-held-out labels were used.')


## 09 — Harmonized fold-safe normalization, identical training and selection

For **both** fusion arms: encoder = ResNet18; initialization = frozen NB02 RGB U-Net, with zeroed additional channels; AdamW LR 8×10⁻⁵, batch 6, 90 steps/epoch, at most 14 epochs, patience 4; exactly the same cached pixels, spatially blocked inner train/selection roles, sampler and augmentation. Only RGB+DSM extra channels use training-fold standardized DSM features. Model-selection metric is equal-site mean BCE on **exhaustive-only** inner validation; an unfused RGB baseline is eligible in both arms. **No external Yoko special case.**


In [ ]:
# 09 — Resumable RGB+DSM and RGB+MS early-fusion U-Net training.
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
import torch,torch.nn as nn,torch.nn.functional as F,copy,gc
DEVICE09=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE09.type=='cuda','Select Colab GPU before running 09'
torch.manual_seed(20261117)
BATCH09=6;EPOCHS09=14;STEPS09=90;PATIENCE09=4;LR09=8e-5
MODEL_OUT09=OUT/'09_harmonized_training'
MODEL_OUT09.mkdir(parents=True,exist_ok=True)
assert TRAINING08.is_file(),'Run 08A, or load the completed joint-chip manifest'
MANIFEST08=pd.read_csv(TRAINING08)
assert set(MANIFEST08.fold)=={0,1,2}
def load_fold09(fold):
    out=[]
    for rec in MANIFEST08.loc[MANIFEST08.fold.eq(fold)].itertuples(index=False):
        with np.load(rec.path,allow_pickle=False) as z:
            assert str(z['signature'].item())==rec.signature
            vals={k:z[k] for k in ['rgb','dsm','ms','y','w','source','site','role']}
        for j in range(len(vals['y'])):
            q={k:vals[k][j] for k in vals}
            q['site']=str(q['site']);q['role']=str(q['role'])
            out.append(q)
    train=[q for q in out if q['role']=='inner_train' and q['w'].sum()>=50]
    val=[q for q in out if q['role']=='inner_val_candidate' and q['w'].sum()>=50]
    assert train and val,(fold,len(train),len(val))
    assert all(not np.any((q['source']==1)&(q['w']>0)) for q in val)
    return train,val
def dsm_norm09(train):
    # Pixels are subsampled regularly to keep the stats calculation bounded.
    ss=np.zeros(10,np.float64);sq=np.zeros(10,np.float64);count=0
    for q in train:
        d=q['dsm'].astype('float32')[:,::8,::8]
        mask=q['w'][::8,::8].astype(bool)
        if not mask.any():continue
        vals=d[:,mask]
        ss+=vals.sum(axis=1);sq+=(vals.astype('float64')**2).sum(axis=1)
        count+=vals.shape[1]
    assert count>=1000,count
    mean=ss/count
    sd=np.sqrt(np.maximum(sq/count-mean**2,1e-4))
    return mean.astype('float32'),sd.astype('float32')
class HarmonizedDataset09(Dataset):
    def __init__(self,items,modality,mu,sd,augment=False):
        self.items=items;self.modality=modality;self.mu=mu[:,None,None]
        self.sd=sd[:,None,None];self.augment=augment
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        q=self.items[i]
        rgb=q['rgb'].astype('float32').transpose(2,0,1)/255.
        if self.modality=='DSM':
            add=np.clip((q['dsm'].astype('float32')-self.mu)/self.sd,-6,6)
        else:
            add=q['ms'].astype('float32')
        x=np.concatenate([rgb,add])
        y=q['y'][None].astype('float32')
        w=q['w'][None].astype('float32')
        if self.augment:
            k=int(torch.randint(0,4,(1,)).item())
            x=np.rot90(x,k,axes=(1,2));y=np.rot90(y,k,axes=(1,2));w=np.rot90(w,k,axes=(1,2))
            if torch.rand(1).item()<.5:
                x=x[:,:,::-1];y=y[:,:,::-1];w=w[:,:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x)),
            torch.from_numpy(np.ascontiguousarray(y)),
            torch.from_numpy(np.ascontiguousarray(w)),q['site'])
def new_model09(fold,n_extra):
    original=torch.load(RGB_CKPT08[fold],map_location='cpu',weights_only=False)
    assert original['version']=='V4.007' and int(original['fold'])==fold
    base=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
    base.load_state_dict({k[4:]:v for k,v in original['model'].items()},strict=True)
    fusion=smp.Unet(encoder_name='resnet18',encoder_weights=None,
                    in_channels=3+n_extra,classes=1)
    state={k:v.detach().cpu().clone() for k,v in base.state_dict().items()}
    conv=state['encoder.conv1.weight']
    state['encoder.conv1.weight']=torch.cat([conv,torch.zeros(
        (conv.shape[0],n_extra,*conv.shape[2:]),dtype=conv.dtype)],dim=1)
    fusion.load_state_dict(state,strict=True)
    return base.to(DEVICE09).eval().requires_grad_(False),fusion.to(DEVICE09)
def run_epoch09(model,loader,optimizer=None,rgb_only=False):
    training=optimizer is not None
    model.train(training)
    buckets={}
    for x,y,w,sites in loader:
        x=x.to(DEVICE09,non_blocking=True)
        y=y.to(DEVICE09);w=w.to(DEVICE09)
        if rgb_only:x=x[:,:3]
        if not w.any():continue
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                logits=model(x)
            loss=F.binary_cross_entropy_with_logits(logits.float(),y,reduction='none')
            obj=(loss*w).sum()/w.sum().clamp_min(1.)
            if training:
                optimizer.zero_grad(set_to_none=True)
                obj.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),2.)
                optimizer.step()
        for j,site in enumerate(sites):
            weight=w[j:j+1];denom=weight.sum().item()
            if denom>0:
                val=float((loss[j:j+1]*weight).sum().item()/denom)
                buckets.setdefault(site,[]).append(val)
    return float(np.mean([np.mean(q) for q in buckets.values()])) if buckets else np.inf
def train_arm09(fold,modality,train,val,mu,sd):
    target=MODEL_OUT09/f'NB05_V4_006_RGB_{modality}_fold{fold}_best.pt'
    if target.is_file():
        ck=torch.load(target,map_location='cpu',weights_only=False)
        assert ck['version']=='V4.006' and ck['fold']==fold and ck['modality']==modality
        assert ck['base_checkpoint']==str(RGB_CKPT08[fold])
        print('MODEL CHECKPOINT HIT',modality,fold,'selected',ck['use_fusion'],flush=True)
        return
    # Matched source pixels, sites, and optimizer across the two modalities.
    torch.manual_seed(20261117+fold)
    counts=pd.Series([q['site'] for q in train]).value_counts().to_dict()
    weights=np.asarray([(2.5 if np.any(q['y']&q['w']) else 1.)/counts[q['site']]
                         for q in train],np.float64)
    sampler=WeightedRandomSampler(weights,num_samples=BATCH09*STEPS09,replacement=True)
    tr=DataLoader(HarmonizedDataset09(train,modality,mu,sd,True),
         sampler=sampler,batch_size=BATCH09,num_workers=0,pin_memory=True)
    va=DataLoader(HarmonizedDataset09(val,modality,mu,sd,False),
         shuffle=False,batch_size=BATCH09,num_workers=0,pin_memory=True)
    rgb,model=new_model09(fold,10 if modality=='DSM' else 4)
    model.eval()
    with torch.inference_mode():
        x=torch.from_numpy(np.random.default_rng(3).random(
            (1,3+(10 if modality=='DSM' else 4),64,64)).astype('float32')).to(DEVICE09)
        err=float((rgb(x[:,:3])-model(x)).abs().max())
        assert err<0.01,('Incorrect zero-extra-channel initialization',err)
    base_bce=run_epoch09(rgb,va,rgb_only=True)
    initial=run_epoch09(model,va)
    assert abs(base_bce-initial)<0.01,(modality,fold,base_bce,initial)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR09,weight_decay=1e-4)
    best_bce=initial
    best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    best_epoch=0;patience=0;hist=[]
    for epoch in range(1,EPOCHS09+1):
        train_bce=run_epoch09(model,tr,optimizer)
        val_bce=run_epoch09(model,va)
        hist.append(dict(modality=modality,fold=fold,epoch=epoch,
            train_macro_bce=train_bce,inner_macro_bce=val_bce,baseline_rgb_bce=base_bce))
        print(modality,'FOLD',fold,'EPOCH',epoch,'train',round(train_bce,4),
              'inner',round(val_bce,4),'RGB',round(base_bce,4),flush=True)
        if np.isfinite(val_bce) and val_bce<best_bce-1e-4:
            best_bce=val_bce
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best_epoch=epoch;patience=0
        else:
            patience+=1
            if patience>=PATIENCE09:break
    selected=bool(best_epoch>0 and best_bce<base_bce-1e-4)
    tmp=target.with_suffix('.partial.pt')
    torch.save(dict(version='V4.006',fold=fold,modality=modality,
        base_checkpoint=str(RGB_CKPT08[fold]),
        architecture=f'smp.Unet_resnet18_in{3+(10 if modality=="DSM" else 4)}',
        state=best_state,norm_mean=mu,norm_std=sd,
        use_fusion=selected,selected_epoch=best_epoch,
        baseline_inner_BCE=base_bce,selected_inner_BCE=best_bce,
        training_count=len(train),selection_count=len(val),
        protocol='NB05_joint_common_support_frozen_NB03_inner_spatial_v1'),tmp)
    os.replace(tmp,target)
    pd.DataFrame(hist).to_csv(MODEL_OUT09/
          f'NB05_inner_history_{modality}_fold{fold}.csv',index=False)
    print('COMPLETED',modality,'fold',fold,'fusion selected',selected,flush=True)
    del rgb,model,tr,va,optimizer
    gc.collect();torch.cuda.empty_cache()
for fold in FOLDS08:
    train,val=load_fold09(fold)
    mu,sd=dsm_norm09(train)
    print('TRAIN MATCHED FOLD',fold,'train',len(train),'inner validation',len(val),
          'sites',dict(pd.Series([q['site'] for q in train]).value_counts()),flush=True)
    for modality in ('DSM','MS'):
        train_arm09(fold,modality,train,val,mu,sd)
    del train,val
    gc.collect();torch.cuda.empty_cache()
print('COMPLETED ALL 6 HARMONIZED FUSION MODELS.')


## 10 — Four strictly harmonized trained U-Nets (RGB / DSM / MS / DSM+MS)

All four modalities are now **fine-tuned on exactly the same training pixels**. Every arm starts from the same frozen historical NB02 RGB pretrained weights, but M0 is also trained with the SAME optimizer, augmentations, epochs and inner validation. Selection considers *trained epochs only (>=1)* and never substitutes an RGB checkpoint. This removes the previously hidden epoch-0 RGB-equivalence for fusion arms. Three spatial outer folds remain strict. Older models/checkpoints remain unchanged for provenance.


In [ ]:
# 09 — Resumable RGB+DSM and RGB+MS early-fusion U-Net training.
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
import torch,torch.nn as nn,torch.nn.functional as F,copy,gc
DEVICE09=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE09.type=='cuda','Select Colab GPU before running 09'
torch.manual_seed(20261117)
BATCH09=6;EPOCHS09=14;STEPS09=90;PATIENCE09=4;LR09=8e-5
MODEL_OUT09=OUT/'09_harmonized_training'
MODEL_OUT09.mkdir(parents=True,exist_ok=True)
assert TRAINING08.is_file(),'Run 08A, or load the completed joint-chip manifest'
MANIFEST08=pd.read_csv(TRAINING08)
assert set(MANIFEST08.fold)=={0,1,2}
def load_fold09(fold):
    out=[]
    for rec in MANIFEST08.loc[MANIFEST08.fold.eq(fold)].itertuples(index=False):
        with np.load(rec.path,allow_pickle=False) as z:
            assert str(z['signature'].item())==rec.signature
            vals={k:z[k] for k in ['rgb','dsm','ms','y','w','source','site','role']}
        for j in range(len(vals['y'])):
            q={k:vals[k][j] for k in vals}
            q['site']=str(q['site']);q['role']=str(q['role'])
            out.append(q)
    train=[q for q in out if q['role']=='inner_train' and q['w'].sum()>=50]
    val=[q for q in out if q['role']=='inner_val_candidate' and q['w'].sum()>=50]
    assert train and val,(fold,len(train),len(val))
    assert all(not np.any((q['source']==1)&(q['w']>0)) for q in val)
    return train,val
def dsm_norm09(train):
    # Pixels are subsampled regularly to keep the stats calculation bounded.
    ss=np.zeros(10,np.float64);sq=np.zeros(10,np.float64);count=0
    for q in train:
        d=q['dsm'].astype('float32')[:,::8,::8]
        mask=q['w'][::8,::8].astype(bool)
        if not mask.any():continue
        vals=d[:,mask]
        ss+=vals.sum(axis=1);sq+=(vals.astype('float64')**2).sum(axis=1)
        count+=vals.shape[1]
    assert count>=1000,count
    mean=ss/count
    sd=np.sqrt(np.maximum(sq/count-mean**2,1e-4))
    return mean.astype('float32'),sd.astype('float32')
class HarmonizedDataset09(Dataset):
    def __init__(self,items,modality,mu,sd,augment=False):
        self.items=items;self.modality=modality;self.mu=mu[:,None,None]
        self.sd=sd[:,None,None];self.augment=augment
    def __len__(self):return len(self.items)
    def __getitem__(self,i):
        q=self.items[i]
        rgb=q['rgb'].astype('float32').transpose(2,0,1)/255.
        if self.modality in ('DSM','BOTH'):
            dsm=np.clip((q['dsm'].astype('float32')-self.mu)/self.sd,-6,6)
        if self.modality=='RGB':add=np.empty((0,*rgb.shape[1:]),dtype='float32')
        elif self.modality=='DSM':add=dsm
        elif self.modality=='MS':add=q['ms'].astype('float32')
        elif self.modality=='BOTH':add=np.concatenate([dsm,q['ms'].astype('float32')])
        else:raise ValueError(self.modality)
        x=np.concatenate([rgb,add])
        y=q['y'][None].astype('float32')
        w=q['w'][None].astype('float32')
        if self.augment:
            k=int(torch.randint(0,4,(1,)).item())
            x=np.rot90(x,k,axes=(1,2));y=np.rot90(y,k,axes=(1,2));w=np.rot90(w,k,axes=(1,2))
            if torch.rand(1).item()<.5:
                x=x[:,:,::-1];y=y[:,:,::-1];w=w[:,:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x)),
            torch.from_numpy(np.ascontiguousarray(y)),
            torch.from_numpy(np.ascontiguousarray(w)),q['site'])
def new_model09(fold,n_extra):
    original=torch.load(RGB_CKPT08[fold],map_location='cpu',weights_only=False)
    assert original['version']=='V4.007' and int(original['fold'])==fold
    base=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1)
    base.load_state_dict({k[4:]:v for k,v in original['model'].items()},strict=True)
    fusion=smp.Unet(encoder_name='resnet18',encoder_weights=None,
                    in_channels=3+n_extra,classes=1)
    state={k:v.detach().cpu().clone() for k,v in base.state_dict().items()}
    conv=state['encoder.conv1.weight']
    state['encoder.conv1.weight']=torch.cat([conv,torch.zeros(
        (conv.shape[0],n_extra,*conv.shape[2:]),dtype=conv.dtype)],dim=1)
    fusion.load_state_dict(state,strict=True)
    return base.to(DEVICE09).eval().requires_grad_(False),fusion.to(DEVICE09)
def run_epoch09(model,loader,optimizer=None,rgb_only=False):
    training=optimizer is not None
    model.train(training)
    buckets={}
    for x,y,w,sites in loader:
        x=x.to(DEVICE09,non_blocking=True)
        y=y.to(DEVICE09);w=w.to(DEVICE09)
        if rgb_only:x=x[:,:3]
        if not w.any():continue
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                logits=model(x)
            loss=F.binary_cross_entropy_with_logits(logits.float(),y,reduction='none')
            obj=(loss*w).sum()/w.sum().clamp_min(1.)
            if training:
                optimizer.zero_grad(set_to_none=True)
                obj.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),2.)
                optimizer.step()
        for j,site in enumerate(sites):
            weight=w[j:j+1];denom=weight.sum().item()
            if denom>0:
                val=float((loss[j:j+1]*weight).sum().item()/denom)
                buckets.setdefault(site,[]).append(val)
    return float(np.mean([np.mean(q) for q in buckets.values()])) if buckets else np.inf
def train_arm09(fold,modality,train,val,mu,sd):
    target=MODEL_OUT09/f'NB05_V4_008_RGB_{modality}_fold{fold}_best.pt'
    if target.is_file():
        ck=torch.load(target,map_location='cpu',weights_only=False)
        assert ck['version']=='V4.008' and ck['fold']==fold and ck['modality']==modality
        assert ck['base_checkpoint']==str(RGB_CKPT08[fold])
        print('MODEL CHECKPOINT HIT',modality,fold,'selected',ck['use_fusion'],flush=True)
        return
    # Matched source pixels, sites, and optimizer across the two modalities.
    torch.manual_seed(20261117+fold)
    counts=pd.Series([q['site'] for q in train]).value_counts().to_dict()
    weights=np.asarray([(2.5 if np.any(q['y']&q['w']) else 1.)/counts[q['site']]
                         for q in train],np.float64)
    sampler=WeightedRandomSampler(weights,num_samples=BATCH09*STEPS09,replacement=True)
    tr=DataLoader(HarmonizedDataset09(train,modality,mu,sd,True),
         sampler=sampler,batch_size=BATCH09,num_workers=0,pin_memory=True)
    va=DataLoader(HarmonizedDataset09(val,modality,mu,sd,False),
         shuffle=False,batch_size=BATCH09,num_workers=0,pin_memory=True)
    rgb,model=new_model09(fold,{'RGB':0,'DSM':10,'MS':4,'BOTH':14}[modality])
    model.eval()
    with torch.inference_mode():
        x=torch.from_numpy(np.random.default_rng(3).random(
            (1,3+({'RGB':0,'DSM':10,'MS':4,'BOTH':14}[modality]),64,64)).astype('float32')).to(DEVICE09)
        err=float((rgb(x[:,:3])-model(x)).abs().max())
        assert err<0.01,('Incorrect zero-extra-channel initialization',err)
    base_bce=run_epoch09(rgb,va,rgb_only=True)
    initial=run_epoch09(model,va)
    assert abs(base_bce-initial)<0.01,(modality,fold,base_bce,initial)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR09,weight_decay=1e-4)
    best_bce=np.inf  # no epoch-0 fallback; select among trained epochs only
    best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    best_epoch=0;patience=0;hist=[]
    for epoch in range(1,EPOCHS09+1):
        train_bce=run_epoch09(model,tr,optimizer)
        val_bce=run_epoch09(model,va)
        hist.append(dict(modality=modality,fold=fold,epoch=epoch,
            train_macro_bce=train_bce,inner_macro_bce=val_bce,baseline_rgb_bce=base_bce))
        print(modality,'FOLD',fold,'EPOCH',epoch,'train',round(train_bce,4),
              'inner',round(val_bce,4),'RGB',round(base_bce,4),flush=True)
        if np.isfinite(val_bce) and val_bce<best_bce-1e-4:
            best_bce=val_bce
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            best_epoch=epoch;patience=0
        else:
            patience+=1
            if patience>=PATIENCE09:break
    assert best_epoch>=1 and np.isfinite(best_bce),('No trained checkpoint selected',fold,modality)
    selected=True # trained arm; not a selection against pretrained RGB
    tmp=target.with_suffix('.partial.pt')
    torch.save(dict(version='V4.008',fold=fold,modality=modality,
        base_checkpoint=str(RGB_CKPT08[fold]),
        architecture=f'smp.Unet_resnet18_in{3+{"RGB":0,"DSM":10,"MS":4,"BOTH":14}[modality]}',
        state=best_state,norm_mean=mu,norm_std=sd,
        use_fusion=selected,selected_epoch=best_epoch,
        baseline_inner_BCE=base_bce,selected_inner_BCE=best_bce,
        training_count=len(train),selection_count=len(val),
        protocol='NB05_joint_common_support_frozen_NB03_inner_spatial_v1'),tmp)
    os.replace(tmp,target)
    pd.DataFrame(hist).to_csv(MODEL_OUT09/
          f'NB05_inner_history_{modality}_fold{fold}.csv',index=False)
    print('COMPLETED',modality,'fold',fold,'fusion selected',selected,flush=True)
    del rgb,model,tr,va,optimizer
    gc.collect();torch.cuda.empty_cache()

for fold in FOLDS08:
    train,val=load_fold09(fold)
    mu,sd=dsm_norm09(train)
    print('M3 FOUR-MODALITY FOLD',fold,'train',len(train),'selection',len(val),flush=True)
    for modality in ('RGB','DSM','MS','BOTH'):
        train_arm09(fold,modality,train,val,mu,sd)
    del train,val
    gc.collect();torch.cuda.empty_cache()
print('ALL FOUR TRAINED ARMS AVAILABLE FOR THREE OUTER FOLDS')


## 11 — Resume-aware full-unit inference without needless SSD copies

New V4.014 checks whether *all* spatial blocks are complete using the original RGB raster header and saved block ledger before staging sensor TIFF files. Completed mosaics are returned from the saved per-unit accumulators in seconds. Partially completed mosaics follow the unchanged V4.013 calculation and can reuse previously staged files from the current Colab runtime. Disable SSD staging (`ENABLE_SSD_STAGE11=False`) when transferring a large new mosaic costs more than its remaining calculations. Run 11 then 12; no retraining.


In [ ]:
# 11 — EXACT ORIGINAL FULL UNIT, FOUR RAW U-NETS, SPATIAL HELD OUT.
from pathlib import Path
from contextlib import ExitStack
from collections import defaultdict, OrderedDict
import json,os,time,hashlib,gc,shutil
import numpy as np,pandas as pd,geopandas as gpd,rasterio,torch
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.strtree import STRtree
from sklearn.metrics import r2_score
# Standalone sensor-window functions (no need to rerun 08A expensive chip extraction).
def bands_vrt08(rgb,groups,stack):
    result={}
    for group in groups:
        result[group]={}
        for band in ('red','green','red_edge','nir'):
            src=stack.enter_context(rasterio.open(local_source11(MS_BANDS[group][band])))
            idx=('red','green','red_edge','nir').index(band)+1 if group=='Baego_MS' else 1
            vrt=stack.enter_context(WarpedVRT(src,crs=rgb.crs,
                transform=rgb.transform,width=rgb.width,height=rgb.height,
                resampling=Resampling.bilinear,dtype='float32'))
            result[group][band]=(vrt,idx)
    return result
def matched_ms08(vrts,win):
    stack=[];supported=[]
    for band in ('red','green','red_edge','nir'):
        ds,idx=vrts[band]
        a=ds.read(idx,window=win,masked=True,out_dtype='float32')
        v=np.asarray(a.filled(0),np.float32)
        ok=(~np.ma.getmaskarray(a))&np.isfinite(v)&(v>0)
        stack.append(np.where(ok,v,0))
        supported.append(ok)
    raw=np.stack(stack)
    valid=np.logical_and.reduce(supported)
    total=raw.sum(axis=0)
    valid &=total>1e-6
    relative=np.where(valid[None],raw/np.maximum(total[None],1e-6),0)
    return relative.astype('float16'),valid

import segmentation_models_pytorch as smp
DEVICE11=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE11.type=='cuda','Select Colab GPU for full-unit inference'
torch.set_num_threads(2)
TILE11=256;STRIDE11=128;BLOCK11=512;BATCH11=12
CACHE_TILES11=160  # ~170 MB: four float32 probability maps + boolean mask / tile
# Stage input files on LOCAL Colab SSD, not Google Drive. Existing output
# signatures remain based on immutable original source paths, so V4.012
# completed block ledgers are reused. No scientific model/feature changes.
ENABLE_SSD_STAGE11=True
SSD_ROOT11=Path('/content/musanga_sensor_stage_v4')
SSD_MIN_FREE_GB11=25
SSD_ROOT11.mkdir(parents=True,exist_ok=True)
def local_source11(original):
    original=Path(original)
    if not ENABLE_SSD_STAGE11:return original
    try:
        relative=original.relative_to(DATA)
    except ValueError:
        return original  # only stage known raster inputs from Data
    dest=SSD_ROOT11/relative
    size=original.stat().st_size
    if dest.is_file() and dest.stat().st_size==size:return dest
    available=shutil.disk_usage(SSD_ROOT11).free
    if available<size+SSD_MIN_FREE_GB11*1e9:
        print('SSD STAGE SKIPPED (disk reserve):',original.name,
              'file GB',round(size/1e9,2),'free GB',round(available/1e9,2),flush=True)
        return original
    dest.parent.mkdir(parents=True,exist_ok=True)
    tmp=dest.with_name(dest.name+'.partial')
    print('STAGING TO LOCAL SSD',original.name,'GB',round(size/1e9,2),flush=True)
    started=time.monotonic()
    # Streaming copy; do not read full source into RAM.
    with open(original,'rb') as inp, open(tmp,'wb') as out:
        shutil.copyfileobj(inp,out,length=16*1024*1024)
    assert tmp.stat().st_size==size
    os.replace(tmp,dest)
    print('SSD STAGED',original.name,'sec',round(time.monotonic()-started,1),flush=True)
    return dest
def pre_stage_mosaic11(mosaic):
    sources=[DATA/'Ortho_RGB'/mosaic,DSM11[mosaic]]
    for group in CHOICES11[mosaic]:
        sources.extend(MS_BANDS[group].values())
    for f in dict.fromkeys(map(Path,sources)):local_source11(f)

MODELS11=('RGB','RGB_DSM','RGB_MS','RGB_DSM_MS')
SAVE11=OUT/'11_four_model_full_unit_mosaic_v4_010'
UNITCACHE11=SAVE11/'units'
UNITCACHE11.mkdir(parents=True,exist_ok=True)
CHOICES11=defaultdict(list)
for g,m in MOSAIC_KEY.items():CHOICES11[m].append(g)
DSM11={Path(r.rgb_path).name:Path(r.dsm_path) for r in PAIRS.itertuples(index=False)}
WEIGHTS11={}
for f in (0,1,2):
    WEIGHTS11[f]=dict(
      RGB=MODEL_OUT09/f'NB05_V4_008_RGB_RGB_fold{f}_best.pt',
      RGB_DSM=MODEL_OUT09/f'NB05_V4_008_RGB_DSM_fold{f}_best.pt',
      RGB_MS=MODEL_OUT09/f'NB05_V4_008_RGB_MS_fold{f}_best.pt',
      RGB_DSM_MS=MODEL_OUT09/f'NB05_V4_008_RGB_BOTH_fold{f}_best.pt')
    assert all(p.is_file() for p in WEIGHTS11[f].values()),WEIGHTS11[f]
# Original frozen support audit: one RGB mosaic per original unit,
# selected using coverage only, never on observed Musanga or predictions.
SUPPORT11=pd.read_csv(OUT/'06_exact_full_unit_support/07_frozen_deduplicated_original_unit_support.csv',
                       dtype={'unit_id':str})
assert not SUPPORT11.duplicated(['kind','unit_id']).any()
SUPPORT11=SUPPORT11.loc[SUPPORT11.common_rgb_dsm_ms_px.gt(0)].copy()
assert len(SUPPORT11)>0
UNIT_GEOMS11={('plot',str(q.Plot_name)):q.geometry for q in PLOTS.itertuples(index=False)}
UNIT_GEOMS11.update({('circle',str(q.circle_id)):q.geometry for q in CIRCLES.itertuples(index=False)
                     if not bool(q.cross_fold_plot_intersection)})
CROWN_P11=gpd.read_file(NB00['exhaustive_plot_crowns']).to_crs(32635)
CROWN_P11=list(CROWN_P11.loc[pd.to_numeric(CROWN_P11.Class_ID,errors='coerce').eq(1)].geometry)
CROWN_C11=list(gpd.read_file(NB00['circle_positive_crowns']).to_crs(32635).geometry)
ROI11=list(ROI.geometry)
TREES11={'plot':STRtree(CROWN_P11),'circle':STRtree(CROWN_C11)}
CGEOM11={'plot':CROWN_P11,'circle':CROWN_C11}
ROI_TREE11=STRtree(ROI11)
h11=np.hanning(258)[1:-1].astype('float32')
HANN11=np.maximum(np.outer(h11,h11),0.01);HANN11/=HANN11.max()
def load_models11(fold):
    nets={}
    for name,channels in [('RGB',3),('RGB_DSM',13),('RGB_MS',7),('RGB_DSM_MS',17)]:
        ck=torch.load(WEIGHTS11[fold][name],map_location='cpu',weights_only=False)
        net=smp.Unet(encoder_name='resnet18',encoder_weights=None,
                     in_channels=channels,classes=1)
        assert ck['version']=='V4.008' and int(ck['fold'])==fold
        assert ck['modality']=={'RGB':'RGB','RGB_DSM':'DSM',
                 'RGB_MS':'MS','RGB_DSM_MS':'BOTH'}[name]
        assert ck['base_checkpoint']==str(RGB_CKPT08[fold])
        assert ck['selected_epoch']>=1 and ck['use_fusion']
        net.load_state_dict(ck['state'],strict=True)
        assert 'norm_mean' in ck
        nets[name]=net.to(DEVICE11).eval().requires_grad_(False)
    return nets
def masks11(ds,win,geom,kind):
    sh=(int(win.height),int(win.width))
    tr=ds.window_transform(win)
    bb=box(*rasterio.windows.bounds(win,ds.transform))
    def burn(geoms):
        if not geoms:return np.zeros(sh,bool)
        return rasterize(((g,1) for g in geoms if g is not None and not g.is_empty),
          out_shape=sh,transform=tr,fill=0,dtype='uint8').astype(bool)
    roi=burn([ROI11[int(k)] for k in ROI_TREE11.query(bb,predicate='intersects')])
    unit=burn([geom])
    tree=TREES11[kind];gs=CGEOM11[kind]
    positive=burn([gs[int(k)] for k in tree.query(bb,predicate='intersects')])
    return unit&roi,positive
def tile_starts11(start,end,limit):
    lo=max(0,((int(start)-TILE11)//STRIDE11)*STRIDE11)
    hi=min(int(end)-1,limit-1)
    vals=[x for x in range(lo,hi+1,STRIDE11)
           if x+TILE11<=limit and x+TILE11>start]
    if start<TILE11:vals.append(0)
    if end>limit-TILE11:vals.append(limit-TILE11)
    return sorted(set(vals))
def signature11(q):
    fold=int(q.fold);mosaic=str(q.mosaic)
    paths=list(WEIGHTS11[fold].values())+[DATA/'Ortho_RGB'/mosaic,DSM11[mosaic],
            DATA/'Ortho_RGB/ROI.shp',DATA/'Ortho_RGB/ROI.dbf',
            DATA/'Ortho_RGB/ROI.shx',Path(NB00['exhaustive_plot_crowns']),
            Path(NB00['circle_positive_crowns']),Path(CONTRACT['folds_csv'])]
    for g in CHOICES11[mosaic]:paths.extend(MS_BANDS[g].values())
    path_stamps=[[str(p),p.stat().st_size,p.stat().st_mtime_ns]
                 for p in sorted(set(map(Path,paths)),key=str)]
    geom=UNIT_GEOMS11[(str(q.kind),str(q.unit_id))]
    payload=dict(recipe='NB05_V4_008_four_strictly_trained_256Hann128_v1',
        unit=str(q.kind)+':'+str(q.unit_id),fold=fold,
        geometry=geom.wkb_hex,files=path_stamps)
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
def predict_block11(ds,dsm,vrts,block,nets,mu,sd,tile_cache=None):
    """Identical Hann/stride predictions, but compute each 256px tile once
    per mosaic and fold and share it between overlapping 512px blocks.
    Cached probabilities remain float32: no changes to inference precision.
    """
    H,W=int(block.height),int(block.width)
    sums=np.zeros((4,H,W),np.float32)
    weight=np.zeros((H,W),np.float32)
    rr=tile_starts11(block.row_off,block.row_off+H,ds.height)
    cc=tile_starts11(block.col_off,block.col_off+W,ds.width)
    positions=[(r,c) for r in rr for c in cc]
    if tile_cache is None:tile_cache=OrderedDict()
    missing=[pos for pos in positions if pos not in tile_cache]
    old_hits=len(positions)-len(missing)
    for first in range(0,len(missing),BATCH11):
        coords=missing[first:first+BATCH11]
        rgb_batch=[];dsm_batch=[];ms_batch=[];ok_batch=[]
        for row,col in coords:
            win=Window(col,row,TILE11,TILE11)
            a=ds.read([1,2,3],window=win,masked=True)
            rgbok=(ds.dataset_mask(window=win)>0)&(~np.ma.getmaskarray(a).any(axis=0))
            feats,dsmok=feature_tile(ds,dsm,win)
            d10=np.clip((feats[3:].astype('float32')-mu[:,None,None])/
                        sd[:,None,None],-6,6)
            candidates=[matched_ms08(vrts[g],win)
                        for g in CHOICES11[Path(ds.name).name]]
            # Exactly the prior acquisition selection and common-valid mask.
            ms4,msok=max(candidates,key=lambda z:int((z[1]&rgbok).sum()))
            joint=rgbok&dsmok&msok
            rgb_batch.append(np.asarray(a.filled(0),np.float32)/255.)
            dsm_batch.append(d10)
            ms_batch.append(ms4.astype('float32'))
            ok_batch.append(joint)
        rgb=np.stack(rgb_batch).astype('float32')
        d10=np.stack(dsm_batch).astype('float32')
        ms=np.stack(ms_batch).astype('float32')
        x={'RGB':rgb,
           'RGB_DSM':np.concatenate([rgb,d10],axis=1),
           'RGB_MS':np.concatenate([rgb,ms],axis=1),
           'RGB_DSM_MS':np.concatenate([rgb,d10,ms],axis=1)}
        with torch.inference_mode(),torch.autocast(device_type='cuda',dtype=torch.float16):
            pred=[torch.sigmoid(nets[name](torch.from_numpy(x[name]).to(DEVICE11)))
                  .float().cpu().numpy()[:,0] for name in MODELS11]
        for j,pos in enumerate(coords):
            # Cache immutable float32 probabilities, not reduced precision.
            tile_cache[pos]=(np.stack([pred[k][j] for k in range(4)]),
                             ok_batch[j].copy())
            tile_cache.move_to_end(pos)
    # Accumulate in exactly the same tile order and float32 arithmetic as
    # V4.011 so already completed block accumulators remain numerically valid.
    for row,col in positions:
        pp,joint=tile_cache[row,col]
        tile_cache.move_to_end((row,col))
        r0=max(row,int(block.row_off));r1=min(row+TILE11,int(block.row_off)+H)
        c0=max(col,int(block.col_off));c1=min(col+TILE11,int(block.col_off)+W)
        if r1<=r0 or c1<=c0:continue
        to=np.s_[r0-int(block.row_off):r1-int(block.row_off),
                  c0-int(block.col_off):c1-int(block.col_off)]
        fr=np.s_[r0-row:r1-row,c0-col:c1-col]
        w=HANN11[fr]*joint[fr]
        for k in range(4):sums[k][to]+=pp[k][fr]*w
        weight[to]+=w
    while len(tile_cache)>CACHE_TILES11:
        tile_cache.popitem(last=False)
    good=weight>0
    for k in range(4):sums[k,good]/=weight[good]
    return sums,good,old_hits,len(missing)


# Optimized mosaic-first evaluation:
# - One 512x512 fixed-grid image prediction PER (outer fold, mosaic, block).
# - Every original plot and circle touching that block receives its own
#   pixel confusion and cover accumulators, all on identical common pixels.
# - Persist group-level accumulator and completed-block ledger atomically.
# - Does not blend/merge reference polygons or change observation unit.
# - Uses a NEW output directory; older per-unit files remain untouched.

GROUPCACHE11=SAVE11/'block_checkpoints'
GROUPCACHE11.mkdir(parents=True,exist_ok=True)
SAVE_EVERY11=5
def fresh_unit11():
    return dict(n_common=0,ref_pos=0,
        counts={m:{k:0 for k in ('tp','tn','fp','fn','pred_pos')} for m in MODELS11},
        soft={m:0.0 for m in MODELS11})
def group_sig11(rows):
    # Signatures include all models, input imagery, reference polygons,
    # output geometry, and algorithm settings. Reuse only exact matches.
    serial=[signature11(q) for q in rows.itertuples(index=False)]
    return hashlib.sha256(json.dumps(dict(
        version='NB05_V4_010_mosaic_first_512px_blocks_no_fallback',
        support=serial,tile=TILE11,stride=STRIDE11,block=BLOCK11,
        models=MODELS11,threshold=.5),sort_keys=True).encode()).hexdigest()
def group_blocks11(rows,ds):
    """Global 512px raster-aligned blocks touching candidate unit bounding boxes."""
    blocks=set()
    for q in rows.itertuples(index=False):
        geom=UNIT_GEOMS11[(str(q.kind),str(q.unit_id))]
        bounds=rasterio.windows.from_bounds(*geom.bounds,transform=ds.transform)
        r0=max(0,int(np.floor(bounds.row_off)))
        c0=max(0,int(np.floor(bounds.col_off)))
        r1=min(ds.height,int(np.ceil(bounds.row_off+bounds.height)))
        c1=min(ds.width,int(np.ceil(bounds.col_off+bounds.width)))
        if r0>=r1 or c0>=c1:continue
        for rr in range((r0//BLOCK11)*BLOCK11,r1,BLOCK11):
            for cc in range((c0//BLOCK11)*BLOCK11,c1,BLOCK11):
                blocks.add((rr,cc))
    return sorted(blocks)
def update_from_block11(state,pred,valid,inside,truth):
    support=inside&valid
    if not support.any():return 0
    y=truth[support]
    state['n_common']+=int(y.size)
    state['ref_pos']+=int(y.sum())
    for k,m in enumerate(MODELS11):
        p=pred[k][support]
        positive=p>=.5
        z=state['counts'][m]
        z['tp']+=int((positive&y).sum())
        z['fp']+=int((positive&~y).sum())
        z['fn']+=int((~positive&y).sum())
        z['tn']+=int((~positive&~y).sum())
        z['pred_pos']+=int(positive.sum())
        state['soft'][m]+=float(np.sum(p,dtype='float64'))
    return int(y.size)
def save_checkpoint11(path,signature,ledger,states):
    tmp=path.with_suffix('.partial')
    tmp.write_text(json.dumps(dict(signature=signature,
        completed_blocks=sorted(ledger),states=states),separators=(',',':')))
    os.replace(tmp,path)
def summarize_mosaic11(states,units,fold,mosaic,elapsed_min=0.0):
    out=[]
    for uid,q in units.items():
        state=states[uid]
        n=int(state['n_common'])
        obs=int(state['ref_pos'])
        rec=dict(kind=str(q.kind),unit_id=str(q.unit_id),
            site=str(q.site),fold=int(fold),mosaic=mosaic,n_common=n,
            reference_cover_pct=100*obs/n if n else np.nan,
            full_common_pct_of_RGB=float(q.common_pct_of_rgb))
        for m in MODELS11:
            z=state['counts'][m]
            for field,num in z.items():rec[f'{m}_{field}']=int(num)
            rec[f'{m}_cover_pct']=100*int(z['pred_pos'])/n if n else np.nan
            rec[f'{m}_soft_cover_pct']=100*state['soft'][m]/n if n else np.nan
        out.append(rec)
    print('MOSAIC COMPLETE',fold,mosaic,'units',len(out),
          'elapsed min',round(elapsed_min,1),flush=True)
    return out

def run_mosaic11(fold,mosaic,rows,nets,mu,sd):
    rows=rows.copy().sort_values(['kind','unit_id'])
    assert len(rows)
    key=lambda q:str(q.kind)+':'+str(q.unit_id)
    units={key(q):q for q in rows.itertuples(index=False)}
    geoms={k:UNIT_GEOMS11[(str(q.kind),str(q.unit_id))] for k,q in units.items()}
    signature=group_sig11(rows)
    # Save separate state per fold and mosaic so existing work is resumable
    # even when a runtime disconnects halfway through a large circle.
    label=hashlib.sha256((str(fold)+'|'+mosaic).encode()).hexdigest()[:14]
    cache=GROUPCACHE11/f'fold{fold}_{label}.json'
    states={k:fresh_unit11() for k in units}
    complete=set()
    if cache.exists():
        saved=json.loads(cache.read_text())
        if saved.get('signature')==signature:
            states=saved['states']
            complete=set(saved['completed_blocks'])
            assert set(states)==set(units)
            print('BLOCK RESUME',fold,mosaic,len(complete),'blocks',flush=True)
        else:
            print('STALE BLOCK CACHE ignored',fold,mosaic,flush=True)
    # Read only the RGB TIFF header first. If every block is already cached,
    # return results immediately: do not stage gigabytes of completed mosaics.
    with rasterio.open(DATA/'Ortho_RGB'/mosaic) as header_ds:
        expected_blocks=group_blocks11(rows,header_ds)
    expected_keys={f'{rr}:{cc}' for rr,cc in expected_blocks}
    assert complete.issubset(expected_keys),('Unexpected cached block coordinates',fold,mosaic)
    if len(complete)==len(expected_keys):
        print('MOSAIC ALREADY COMPLETE — NO SSD TRANSFER:',fold,mosaic,
              'blocks',len(complete),flush=True)
        return summarize_mosaic11(states,units,fold,mosaic,0.0)
    pre_stage_mosaic11(mosaic)
    started=time.monotonic()
    with rasterio.Env(GDAL_CACHEMAX=128):
        with ExitStack() as handles:
            ds=handles.enter_context(rasterio.open(local_source11(DATA/'Ortho_RGB'/mosaic)))
            dsm=handles.enter_context(rasterio.open(local_source11(DSM11[mosaic])))
            vrts=bands_vrt08(ds,CHOICES11[mosaic],handles)
            blocks=group_blocks11(rows,ds)
            print('MOSAIC FOLD',fold,mosaic,'units',len(units),
                  'unique blocks',len(blocks),'cached',len(complete),flush=True)
            # Set up one geometry index for all reference polygons of this mosaic,
            # so a block needs one model pass regardless of how many units overlap.
            keys=list(units)
            unit_geoms=[geoms[k] for k in keys]
            unit_tree=STRtree(unit_geoms)
            newly=0
            tile_cache=OrderedDict()  # per fold/mosaic; never mixes model weights
            hit_total=0;miss_total=0
            for i,(row,col) in enumerate(blocks):
                blockkey=f'{row}:{col}'
                if blockkey in complete:continue
                win=Window(col,row,min(BLOCK11,ds.width-col),
                           min(BLOCK11,ds.height-row))
                bbox=box(*rasterio.windows.bounds(win,ds.transform))
                candidates=[keys[int(j)] for j in
                    unit_tree.query(bbox,predicate='intersects')]
                if not candidates:
                    complete.add(blockkey);continue
                # Burn reference masks BEFORE doing expensive sensor extraction.
                masks={}
                for uid in candidates:
                    u=units[uid]
                    inside,truth=masks11(ds,win,geoms[uid],str(u.kind))
                    if inside.any():masks[uid]=(inside,truth)
                if masks:
                    output,valid,hits,misses=predict_block11(
                        ds,dsm,vrts,win,nets,mu,sd,tile_cache)
                    hit_total+=hits;miss_total+=misses
                    for uid,(inside,truth) in masks.items():
                        update_from_block11(states[uid],output,valid,inside,truth)
                complete.add(blockkey)
                newly+=1
                if newly%SAVE_EVERY11==0:
                    save_checkpoint11(cache,signature,complete,states)
                    print('FOLD',fold,'MOSAIC',mosaic,'blocks',
                          len(complete),'/',len(blocks),
                          'elapsed min',round((time.monotonic()-started)/60,1),
                          'tile reuse',round(hit_total/max(1,hit_total+miss_total),2),
                          flush=True)
            save_checkpoint11(cache,signature,complete,states)
            assert len(complete)==len(blocks),('Incomplete mosaic',fold,mosaic)
    return summarize_mosaic11(states,units,fold,mosaic,(time.monotonic()-started)/60)
RESULTS11=[]
for fold in (0,1,2):
    selected=SUPPORT11.loc[SUPPORT11.fold.eq(fold)]
    if selected.empty:continue
    nets=load_models11(fold)
    ck=torch.load(WEIGHTS11[fold]['RGB_DSM'],map_location='cpu',weights_only=False)
    mu=np.asarray(ck['norm_mean'],np.float32)
    sd=np.asarray(ck['norm_std'],np.float32)
    for name in ('RGB_MS','RGB_DSM_MS'):
        cc=torch.load(WEIGHTS11[fold][name],map_location='cpu',weights_only=False)
        assert np.allclose(mu,cc['norm_mean']) and np.allclose(sd,cc['norm_std'])
    del ck,cc
    for mosaic,rows in selected.groupby('mosaic',sort=True):
        RESULTS11.extend(run_mosaic11(fold,mosaic,rows,nets,mu,sd))
        pd.DataFrame(RESULTS11).to_csv(SAVE11/'11_partial.csv',index=False)
    del nets
    torch.cuda.empty_cache();gc.collect()
DF11=pd.DataFrame(RESULTS11)
assert len(DF11)==len(SUPPORT11)
assert not DF11.duplicated(['kind','unit_id']).any()
DF11.to_csv(SAVE11/'11_original_units_all_four_raw_models.csv',index=False)
print('COMPLETE original units:',len(DF11),
      'with common pixels',int(DF11.n_common.gt(0).sum()))
display(DF11.groupby(['site','kind']).agg(n=('unit_id','size'),
          common_pixels=('n_common','sum')).round(1))


## 12 — Fractional-cover accuracy (primary) and segmentation metrics (secondary)

Results are **unit-weighted**, not pixel-weighted, with separate plot and circle tables and per-site breakdown; equal-site averages prevent Yangambi dominating the global comparison. Paired model predictions always share exactly the same masks. We report hard-threshold (0.5) and soft probability-sum fractional cover. Segmentation F1/IoU/precision/recall are computed from held-out pixel confusion per unit, with macro unit and pixel-micro summaries. High-cover MOS subset defined by **reference Musanga cover > 40%**, not predictions. RPD = SD(observed cover)/RMSE; RPIQ = IQR(observed cover)/RMSE; no artificial scores when variation is insufficient. Save CSV figures and interpretations, with all arms have genuinely trained checkpoints and no fallback.


In [ ]:
# 12 — One honest paired 4-model comparison, all spatially held-out predictions.
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.metrics import r2_score
if 'DF11' not in globals():
    DF11=pd.read_csv(SAVE11/'11_original_units_all_four_raw_models.csv',
                     dtype={'unit_id':str})
DATA12=DF11[DF11.n_common.gt(0)].copy()
assert not DATA12.empty
MODELS12=('RGB','RGB_DSM','RGB_MS','RGB_DSM_MS')
def metrics12(frame,m,soft=False):
    y=frame.reference_cover_pct.to_numpy(float)
    col=f'{m}_soft_cover_pct' if soft else f'{m}_cover_pct'
    p=frame[col].to_numpy(float)
    difference=p-y
    rmse=float(np.sqrt(np.mean(difference**2)))
    sd=float(np.std(y,ddof=1)) if len(y)>1 else np.nan
    iqr=float(np.subtract(*np.percentile(y,[75,25]))) if len(y)>1 else np.nan
    tp=frame[f'{m}_tp'].sum();fp=frame[f'{m}_fp'].sum()
    fn=frame[f'{m}_fn'].sum()
    recall=tp/(tp+fn) if tp+fn else np.nan
    precision=tp/(tp+fp) if tp+fp else np.nan
    unit_f1=[];unit_iou=[]
    for q in frame.itertuples(index=False):
        a=getattr(q,f'{m}_tp');b=getattr(q,f'{m}_fp');c=getattr(q,f'{m}_fn')
        unit_f1.append(2*a/(2*a+b+c) if 2*a+b+c else np.nan)
        unit_iou.append(a/(a+b+c) if a+b+c else np.nan)
    return dict(n_units=len(frame),R2=float(r2_score(y,p)) if len(y)>1 and sd>0 else np.nan,
       MAE_pp=float(np.mean(abs(difference))),RMSE_pp=rmse,
       bias_pp=float(np.mean(difference)),
       RPD=float(sd/rmse) if rmse>0 and sd>0 else np.nan,
       RPIQ=float(iqr/rmse) if rmse>0 and iqr>0 else np.nan,
       F1_micro=float(2*tp/(2*tp+fp+fn)) if 2*tp+fp+fn else np.nan,
       IoU_micro=float(tp/(tp+fp+fn)) if tp+fp+fn else np.nan,
       precision_micro=float(precision),recall_micro=float(recall),
       F1_unit_macro=float(np.nanmean(unit_f1)) if np.isfinite(unit_f1).any() else np.nan,
       IoU_unit_macro=float(np.nanmean(unit_iou)) if np.isfinite(unit_iou).any() else np.nan)
REPORT12=[]
for kind in ('plot','circle'):
    K=DATA12[DATA12.kind.eq(kind)]
    if K.empty:continue
    for site in ['ALL_UNITS','Baego','MOS','Yangambi','Yoko']:
        subset=K if site=='ALL_UNITS' else K[K.site.eq(site)]
        if subset.empty:continue
        for m in MODELS12:
            for soft in (False,True):
                REPORT12.append(dict(kind=kind,site=site,model=m,
                     cover_type='soft' if soft else 'hard_0.5',
                     **metrics12(subset,m,soft=soft)))
REPORT12=pd.DataFrame(REPORT12)
REPORT12.to_csv(SAVE11/'12_cover_and_segmentation_metrics.csv',index=False)
display(REPORT12[REPORT12.cover_type.eq('hard_0.5')].round(3))
SITE12=REPORT12[(REPORT12.site.isin(['Baego','MOS','Yangambi','Yoko']))&
                 REPORT12.cover_type.eq('hard_0.5')]
sitebalanced=SITE12.groupby(['kind','model'])[['MAE_pp','RMSE_pp','bias_pp','F1_unit_macro']].mean().reset_index()
sitebalanced.to_csv(SAVE11/'12_site_balanced_metrics.csv',index=False)
print('EQUAL-SITE MACRO AVERAGES (each site equal weight):')
display(sitebalanced.round(3))
high=DATA12[(DATA12.kind=='plot')&(DATA12.site=='MOS')&
            (DATA12.reference_cover_pct>40)]
if len(high):
    H=pd.DataFrame([dict(model=m,n_high=len(high),**metrics12(high,m)) for m in MODELS12])
    H.to_csv(SAVE11/'12_MOS_reference_high_cover_gt40.csv',index=False)
    print('MOS HIGH COVER (>40% ORIGINAL REFERENCE):')
    display(H.round(3))
else:print('No MOS plot above 40% reference cover among common-valid units.')
# Primary visual: identical held-out original plots; site-colored, same axes.
plot=DATA12[DATA12.kind.eq('plot')]
fig,axs=plt.subplots(2,2,figsize=(11,10),sharex=True,sharey=True)
colors={'Baego':'tab:blue','MOS':'tab:orange',
        'Yangambi':'tab:green','Yoko':'tab:red'}
for ax,m in zip(axs.flat,MODELS12):
    for site,q in plot.groupby('site'):
        ax.scatter(q.reference_cover_pct,q[f'{m}_cover_pct'],s=27,
                   alpha=.73,c=colors.get(site,'gray'),label=site)
    ax.plot([0,100],[0,100],'--',c='0.5',lw=1)
    ax.set(xlim=(0,100),ylim=(0,100),title=m,
           xlabel='Held-out original plot reference cover (%)',
           ylabel='Predicted cover (%)')
axs.flat[0].legend(loc='upper left',fontsize=9)
fig.suptitle('All four genuine U-Nets — matched common-valid pixels, spatially held-out plots')
fig.tight_layout()
fig.savefig(SAVE11/'12_four_model_cover_scatter.png',dpi=190,bbox_inches='tight')
plt.show()
print('ALL OUTPUTS',SAVE11)
print('NOTE: raw learned fusion models evaluated regardless of historical use_fusion flags.')


## 13 — Quick read-only profiling plus two-pass benchmark

Paste/run only this cell into the stopped current Colab session (block 11 functions must remain defined). Profiles one original Yangambi 512px block, repeats that block to distinguish first-pass and warm-cache costs, verifies identical valid masks and probabilities to 2e-4 tolerance, and prints timings for DSM features, multispectral reads, GPU model passes, and residual read/assembly/blending work. It does **not** stage imagery, write checkpoint state, or train anything. After sharing the table, we can target the actual bottleneck.


In [ ]:
# 13 — READ-ONLY one-block diagnostic. Run in the SAME live Colab session
# after STOPPING block 11. Does NOT train, stage new data or write checkpoints.
import time, gc
from pathlib import Path
from contextlib import ExitStack
from collections import OrderedDict, defaultdict
import numpy as np, pandas as pd, rasterio, torch
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from shapely.geometry import box
required=['SUPPORT11','UNIT_GEOMS11','WEIGHTS11','load_models11',
          'predict_block11','feature_tile','matched_ms08',
          'group_blocks11','CHOICES11','MS_BANDS','DSM11','BATCH11']
missing=[s for s in required if s not in globals()]
assert not missing, ('Missing section-11 definitions; paste into interrupted Colab runtime',missing)
assert torch.cuda.is_available()
FOLD_P=0
MOSAIC_P='Yangambi_georef.tif'
rows_P=SUPPORT11[(SUPPORT11.fold==FOLD_P)&(SUPPORT11.mosaic==MOSAIC_P)]
assert len(rows_P)
# Use previously staged files only; never copy another byte to SSD.
def passive_path_P(original):
    original=Path(original)
    try:
        local=Path('/content/musanga_sensor_stage_v4')/original.relative_to(DATA)
        if local.is_file() and local.stat().st_size==original.stat().st_size:
            return local
    except ValueError:
        pass
    return original
# Temporarily intercept only sensor helper calls and torch forward passes;
# restore everything in finally. This profiles THE ORIGINAL block function.
orig_ft_P=feature_tile
orig_ms_P=matched_ms08
orig_local_P=local_source11
stats_P=defaultdict(float)
handles_P=[]
def now_P():
    torch.cuda.synchronize()
    return time.perf_counter()
def timed_feature_P(*args,**kwargs):
    start=time.perf_counter()
    v=orig_ft_P(*args,**kwargs)
    stats_P['DSM feature_tile']+=time.perf_counter()-start
    return v
def timed_ms_P(*args,**kwargs):
    start=time.perf_counter()
    v=orig_ms_P(*args,**kwargs)
    stats_P['MS four-band read/resample']+=time.perf_counter()-start
    return v
def forward_before_P(model,args):
    model._musanga_profile_start=now_P()
def forward_after_P(model,args,output):
    stats_P['4 U-Net forward passes']+=now_P()-model._musanga_profile_start
try:
    globals()['feature_tile']=timed_feature_P
    globals()['matched_ms08']=timed_ms_P
    globals()['local_source11']=passive_path_P
    nets_P=load_models11(FOLD_P)
    for net in nets_P.values():
        handles_P.append(net.register_forward_pre_hook(forward_before_P))
        handles_P.append(net.register_forward_hook(forward_after_P))
    ck_P=torch.load(WEIGHTS11[FOLD_P]['RGB_DSM'],map_location='cpu',weights_only=False)
    mu_P=np.asarray(ck_P['norm_mean'],np.float32)
    sd_P=np.asarray(ck_P['norm_std'],np.float32)
    del ck_P
    with rasterio.Env(GDAL_CACHEMAX=128):
        with ExitStack() as stack:
            rgb_P=stack.enter_context(rasterio.open(passive_path_P(DATA/'Ortho_RGB'/MOSAIC_P)))
            dsm_P=stack.enter_context(rasterio.open(passive_path_P(DSM11[MOSAIC_P])))
            # Same RGB-grid VRT settings as the production section.
            vrts_P=bands_vrt08(rgb_P,CHOICES11[MOSAIC_P],stack)
            blocks_P=group_blocks11(rows_P,rgb_P)
            win_P=None
            for idx in sorted(range(len(blocks_P)),key=lambda x:abs(x-len(blocks_P)//2)):
                rr,cc=blocks_P[idx]
                candidate=Window(cc,rr,min(BLOCK11,rgb_P.width-cc),
                                 min(BLOCK11,rgb_P.height-rr))
                bounds=box(*rasterio.windows.bounds(candidate,rgb_P.transform))
                if any(UNIT_GEOMS11[(str(z.kind),str(z.unit_id))].intersects(bounds)
                       for z in rows_P.itertuples(index=False)):
                    win_P=candidate
                    break
            assert win_P is not None
            print('Profiling Yangambi fold 0 block:',int(win_P.row_off),
                  int(win_P.col_off),'shape',int(win_P.height),int(win_P.width),
                  'GPU batch',BATCH11,flush=True)
            before_P=now_P()
            pred_P,mask_P,*_=predict_block11(
                rgb_P,dsm_P,vrts_P,win_P,nets_P,mu_P,sd_P,OrderedDict())
            total_P=now_P()-before_P
            print('Matched valid pixels:',int(mask_P.sum()),flush=True)
            # Same-source second run: warm-cache benchmark + identical-output test.
            first_stats_P=dict(stats_P)
            stats_P.clear()
            t2=now_P()
            pred2_P,mask2_P,*_=predict_block11(
                rgb_P,dsm_P,vrts_P,win_P,nets_P,mu_P,sd_P,OrderedDict())
            warm_total_P=now_P()-t2
            assert np.array_equal(mask_P,mask2_P), 'Second pass mask changed'
            diff_P=float(np.max(np.abs(pred_P-pred2_P)))
            assert diff_P<=2e-4, ('Repeated prediction changed',diff_P)
            print('BENCHMARK same block: first',round(total_P,2),
                  's; second (warm)',round(warm_total_P,2),
                  's; maximum probability difference',diff_P,flush=True)
finally:
    globals()['feature_tile']=orig_ft_P
    globals()['matched_ms08']=orig_ms_P
    globals()['local_source11']=orig_local_P
    for hook in handles_P:hook.remove()
    if 'nets_P' in locals():del nets_P
    torch.cuda.empty_cache()
    gc.collect()
# GPU component uses synchronized hooks; the residual captures RGB reads,
# transfers, input assembly, Hann blending and Python overhead.
tracked_P=sum(stats_P.values())
stats_P['RGB+assembly+Hann+overhead']=max(0.,warm_total_P-tracked_P)
table_P=pd.DataFrame([{'component':k,'seconds':round(v,2),
    'percent':round(100*v/warm_total_P,1)}
    for k,v in stats_P.items()]).sort_values('seconds',ascending=False)
display(table_P)
print('WARM BLOCK:',round(warm_total_P,2),'seconds | FIRST:',round(total_P,2),'seconds')
print('Dominant component:',table_P.iloc[0].component)
print('No cache ledgers, training checkpoints or raster outputs modified.')


## 14 — Diagnose which DSM structural operation consumes 16 seconds

Run **only section 14** in the already initialized and stopped Colab runtime. It benchmarks two native 256 × 256 Yangambi tiles and separates `local_feature_tile` from `context_feature_tile`, with an assembly equivalence check. It **does not** load U-Nets, write checkpoint ledgers, stage mosaics or perform full-unit inference. This is the final diagnostic needed before a targeted, validated feature-computation optimization.


In [ ]:
# 14 — Deep DSM subcomponent profiling on TWO original 256x256 Yangambi tiles.
# No training, no inference, no block ledgers, no model load.
# Run in SAME Colab session as 13 (or after section 08 + initialized 11 defs).
from contextlib import ExitStack
from pathlib import Path
from rasterio.windows import Window
import rasterio,time,numpy as np,pandas as pd
needed=['local_feature_tile','context_feature_tile','feature_tile','PLOTS',
        'DATA','DSM11','UNIT_GEOMS11','SUPPORT11','group_blocks11']
assert all(name in globals() for name in needed),'Need existing section 08 and 11 definitions'
mosaic14='Yangambi_georef.tif'
unit14=SUPPORT11[(SUPPORT11.fold==0)&(SUPPORT11.mosaic==mosaic14)]
assert len(unit14)
def maybe_local14(path):
    path=Path(path)
    root=Path('/content/musanga_sensor_stage_v4')
    try:
        staged=root/path.relative_to(DATA)
        if staged.is_file() and staged.stat().st_size==path.stat().st_size:return staged
    except ValueError:pass
    return path
records14=[]
with ExitStack() as stack:
    ds14=stack.enter_context(rasterio.open(maybe_local14(DATA/'Ortho_RGB'/mosaic14)))
    dsm14=stack.enter_context(rasterio.open(maybe_local14(DSM11[mosaic14])))
    blocks14=group_blocks11(unit14,ds14)
    assert blocks14
    for jj in [len(blocks14)//3,2*len(blocks14)//3]:
        rr,cc=blocks14[jj]
        win=Window(cc,rr,256,256)
        # Both are the SAME sensor grid and 256px shape as production.
        times={}
        t0=time.perf_counter()
        xyz,valid=local_feature_tile(ds14,dsm14,win)
        times['local_feature_tile']=time.perf_counter()-t0
        t0=time.perf_counter()
        ctx=context_feature_tile(ds14,dsm14,win)
        times['context_feature_tile']=time.perf_counter()-t0
        t0=time.perf_counter()
        entire,joint=feature_tile(ds14,dsm14,win)
        times['full_feature_tile']=time.perf_counter()-t0
        assert np.array_equal(valid,joint)
        assert np.array_equal(entire[:6],xyz)
        assert np.allclose(entire[6:],np.where(joint[None],ctx,0),atol=1e-6)
        for name,secs in times.items():
            records14.append(dict(block_id=jj,component=name,seconds=secs,
                              rgb_pixel_m=abs(ds14.transform.a),
                              dsm_pixel_m=abs(dsm14.transform.a)))
display(pd.DataFrame(records14).pivot(index='block_id',columns='component',values='seconds').round(3))
print('This identifies whether high-resolution local DSM filters or coarse 60m contextual features dominate.')
print('Numerical assembly verification passed. No source or checkpoints modified.')


## 15 — Verify a faster DSM context resampling implementation

The previous profile identified context extraction as the main cost. This read-only microbenchmark tests a **single seven-band resampling operation** against seven individual GDAL warps on four native-resolution Yangambi tiles. It reports original and alternative times and their exact numerical differences. **No replacement is activated** until both speed and feature equivalence are established. Run only section 15 in your existing initialized Colab session. No training, block checkpoints, or full-mosaic inference.


In [ ]:
# 15 — DSM context micro-optimization ONLY. No change to production functions.
# Reuses section 08 definitions and previously staged Yangambi data.
from pathlib import Path
from contextlib import ExitStack
import time, numpy as np, pandas as pd, rasterio
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.transform import Affine
from math import floor,ceil
assert all(s in globals() for s in ['context_feature_tile','_box_mean',
    '_annular_mean','CONTEXT_BANDS','CONTEXT_MAX_RADIUS_M',
    'CONTEXT_RESOLUTION_M','DATA','DSM11','SUPPORT11','group_blocks11'])
def only_existing_local18(path):
    path=Path(path)
    try:
        dst=Path('/content/musanga_sensor_stage_v4')/path.relative_to(DATA)
        if dst.exists() and dst.stat().st_size==path.stat().st_size:return dst
    except ValueError:pass
    return path

def context_feature_tile_multiband18(rgb,dsm,win):
    """Seven spatial contextual features at native RGB grid, coarse computation at 1 m."""
    bounds=rasterio.windows.bounds(win,rgb.transform)
    xmin,ymin,xmax,ymax=bounds
    pad=CONTEXT_MAX_RADIUS_M+4*CONTEXT_RESOLUTION_M
    unit=CONTEXT_RESOLUTION_M
    left=floor((xmin-pad)/unit)*unit
    right=ceil((xmax+pad)/unit)*unit
    top=ceil((ymax+pad)/unit)*unit
    bottom=floor((ymin-pad)/unit)*unit
    w=int(round((right-left)/unit));h=int(round((top-bottom)/unit))
    assert 0<w<=8192 and 0<h<=8192,(h,w,'Unexpected context window')
    tr=Affine(unit,0,left,0,-unit,top)
    with WarpedVRT(dsm,crs=rgb.crs,transform=tr,width=w,height=h,
                   resampling=Resampling.bilinear,src_nodata=dsm.nodata,nodata=np.nan) as vrt:
        raster=vrt.read(1,masked=True)
    z=np.asarray(raster.data,dtype=np.float32)
    valid=(~np.ma.getmaskarray(raster))&np.isfinite(z)
    # Preserve relative heights but remove global offset before variance estimation.
    reference=float(np.median(z[valid])) if np.any(valid) else 0.
    relative=np.where(valid,z-reference,0.).astype(np.float32)
    local,_=_box_mean(relative,valid,3)
    surfaces=[];fractions=[]
    for inner,outer in ((5,15),(15,30),(30,60)):
        surround,frac=_annular_mean(relative,valid,inner,outer)
        surfaces.append(np.where(frac>=.6,local-surround,0.).astype(np.float32))
        fractions.append(frac)
    rough=[]
    for radius in (15,30,60):
        mean,cov=_box_mean(relative,valid,radius)
        second,_=_box_mean(relative*relative,valid,radius)
        rough.append(np.where(cov>=.6,np.sqrt(np.maximum(second-mean*mean,0.)),0.).astype(np.float32))
    coarse=np.stack(surfaces+rough+[fractions[2]],axis=0)
    dst_transform=rgb.window_transform(win)
    out=np.zeros((len(CONTEXT_BANDS),int(win.height),int(win.width)),dtype=np.float32)
    # Smooth resampling of contextual maps. No extrapolation beyond coarse coverage.
    # One multi-band GDAL warp instead of 7 separate warps.
    # Same source raster, destination grid, interpolation and no-data semantics.
    reproject(source=coarse,destination=out,
              src_transform=tr,src_crs=rgb.crs,dst_transform=dst_transform,
              dst_crs=rgb.crs,resampling=Resampling.bilinear,
              src_nodata=None,dst_nodata=-9999.)
    # 0 is a legitimate relative-elevation/roughness value. It must NOT be
    # used as destination NoData: GDAL may perturb valid zeros to 1.4013e-45.
    # Convert only pixels explicitly initialized to -9999 back to 0 in the
    # final feature tensor (the contextual coverage band carries availability).
    out[out==-9999.]=0.
    assert np.isfinite(out).all()
    return out



mosaic18='Yangambi_georef.tif'
rows18=SUPPORT11[(SUPPORT11.fold==0)&(SUPPORT11.mosaic==mosaic18)]
assert len(rows18)
results18=[]
with ExitStack() as stack:
    rgb18=stack.enter_context(rasterio.open(only_existing_local18(DATA/'Ortho_RGB'/mosaic18)))
    dsm18=stack.enter_context(rasterio.open(only_existing_local18(DSM11[mosaic18])))
    blocks18=group_blocks11(rows18,rgb18)
    # Check four distributed native-grid tiles and repeats for timing stability.
    sample18=[blocks18[min(len(blocks18)-1,int(i*(len(blocks18)-1)/3))] for i in range(4)]
    for j,(row,col) in enumerate(sample18):
        win18=Window(col,row,256,256)
        t0=time.perf_counter()
        original18=context_feature_tile(rgb18,dsm18,win18)
        old_time18=time.perf_counter()-t0
        t0=time.perf_counter()
        test18=context_feature_tile_multiband18(rgb18,dsm18,win18)
        new_time18=time.perf_counter()-t0
        absdiff=np.abs(original18-test18)
        maxdiff=float(absdiff.max())
        neq=int(np.count_nonzero(absdiff>1e-5))
        results18.append(dict(tile=j,original_s=old_time18,multiband_s=new_time18,
            max_abs_diff=maxdiff,pixels_diff_gt_1e5=neq,
            speedup=old_time18/max(new_time18,1e-9)))
        print('CHECK',j,'original',round(old_time18,3),'s','single warp',
            round(new_time18,3),'s','max feature diff',maxdiff,flush=True)
df18=pd.DataFrame(results18)
display(df18.round(5))
print('Median context speedup:',round(float(df18.speedup.median()),2),'x')
print('Strict feature equivalence threshold <=1e-5:',
      bool((df18.max_abs_diff<=1e-5).all()))
if not (df18.max_abs_diff<=1e-5).all():
    print('NOT approved: this optimization changes contextual feature values.')
else:
    print('Numerically equivalent on tested tiles; benchmark only, not enabled in inference.')
print('NO changes to main context_feature_tile, blocks, ledgers, weights, or image files.')
